================================================================================
ANÁLISE CAPÍTULO 4 - TESTES DE CURTO-CIRCUITO
Sistema IEEE 34 Barras - MRT (SWER) vs T2F
Versão Profissional para Tese de Doutorado
================================================================================
Autor: Leonardo Santos
Data: Janeiro 2026
================================================================================

In [1]:
import h5py
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import matplotlib.gridspec as gridspec
import pandas as pd
# --- 1. CONFIGURAÇÕES GERAIS E ESTILO ---
FREQUENCIA_REDE = 60 # Hz

def cm_to_inch(value): return value / 2.54

# [ATUALIZADO] Configurações globais para garantir fundo branco
# [ATUALIZADO] Configurações para fundo branco E TEXTO PRETO
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman"],
    "font.size": 12,
    "figure.figsize": (cm_to_inch(16), cm_to_inch(10)),

    # Cores de Fundo
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
    "savefig.transparent": False,

    # Cores de Texto e Linhas (Tudo Preto)
    "text.color": "black",
    "axes.labelcolor": "black",
    "xtick.color": "black",
    "ytick.color": "black",
    "axes.edgecolor": "black"
})

# ==============================================================================
# SEÇÃO 2: FUNÇÕES MATEMÁTICAS AUXILIARES
# ==============================================================================

def calcular_rms_movel(sinal, tempo, freq_rede=60):
    if len(tempo) < 2: return np.zeros_like(sinal)
    dt = tempo[1] - tempo[0]
    if dt <= 0: return np.zeros_like(sinal)
    fs = 1 / dt
    janela = int(fs / freq_rede)
    if janela < 1: janela = 1
    sinal_quadrado = sinal ** 2
    janela_media = np.ones(janela) / janela
    return np.sqrt(np.convolve(sinal_quadrado, janela_media, mode='same'))

def calcular_fft_para_plot(sinal, tempo):
    dt = tempo[1] - tempo[0]
    n = len(sinal)
    fhat = np.fft.fft(sinal)
    freqs = np.fft.fftfreq(n, d=dt)
    magnitudes = 2 * np.abs(fhat) / n
    mask = freqs >= 0
    return freqs[mask], magnitudes[mask]

def calcular_v1_v3_global(sinal, tempo, freq_rede=60):
    dt = tempo[1] - tempo[0]
    n = len(sinal)
    fhat = np.fft.fft(sinal)
    freqs = np.fft.fftfreq(n, d=dt)
    mags = 2 * np.abs(fhat) / n
    idx_60 = np.argmin(np.abs(freqs - freq_rede))
    idx_180 = np.argmin(np.abs(freqs - 3*freq_rede))
    return float(mags[idx_60]), float(mags[idx_180])

def get_imax_envelope_vetor(tempo, dados_tres_fases):
    rms_a = calcular_rms_movel(dados_tres_fases[:, 0], tempo)
    rms_b = calcular_rms_movel(dados_tres_fases[:, 1], tempo)
    rms_c = calcular_rms_movel(dados_tres_fases[:, 2], tempo)
    return np.maximum.reduce([rms_a, rms_b, rms_c])

def extrair_fasor_dinamico(sinal, tempo, freq=60):
    dt = tempo[1] - tempo[0]
    if dt <= 0: return np.zeros_like(sinal, dtype=complex)
    samples_per_cycle = int((1/freq) / dt)
    if samples_per_cycle < 1: samples_per_cycle = 1
    t_window = np.arange(samples_per_cycle) * dt
    kernel_cos = np.cos(2 * np.pi * freq * t_window) * (2/samples_per_cycle)
    kernel_sin = np.sin(2 * np.pi * freq * t_window) * (2/samples_per_cycle)
    real_part = np.convolve(sinal, kernel_cos, mode='same')
    imag_part = np.convolve(sinal, kernel_sin, mode='same')
    return real_part - 1j * imag_part

def calcular_componentes_simetricas_tempo(dados_3fases, tempo, freq=60):
    Va = extrair_fasor_dinamico(dados_3fases[:, 0], tempo, freq)
    Vb = extrair_fasor_dinamico(dados_3fases[:, 1], tempo, freq)
    Vc = extrair_fasor_dinamico(dados_3fases[:, 2], tempo, freq)
    a = np.exp(1j * 2 * np.pi / 3)
    a2 = a**2
    V0 = (Va + Vb + Vc) / 3
    V1 = (Va + a * Vb + a2 * Vc) / 3
    V2 = (Va + a2 * Vb + a * Vc) / 3
    return np.abs(V0), np.abs(V1), np.abs(V2)

# ==============================================================================
# SEÇÃO 3: FUNÇÕES DE PLOTAGEM (ATUALIZADAS PARA FUNDO BRANCO)
# ==============================================================================

def gerar_grafico_fft_espectro(tempo, dados_raw, nome_arquivo_saida, pasta_saida, ylabel, limite_freq_visual=1000):
    freqs_a, mag_a = calcular_fft_para_plot(dados_raw[:, 0], tempo)
    freqs_b, mag_b = calcular_fft_para_plot(dados_raw[:, 1], tempo)
    freqs_c, mag_c = calcular_fft_para_plot(dados_raw[:, 2], tempo)

    plt.figure()
    plt.plot(freqs_a, mag_a, color='red', label='Fase A', linewidth=1.2, alpha=0.8)
    plt.plot(freqs_b, mag_b, color='blue', label='Fase B', linewidth=1.2, alpha=0.8)
    plt.plot(freqs_c, mag_c, color='green', label='Fase C', linewidth=1.2, alpha=0.8)

    plt.xlabel('Frequência (Hz)')
    plt.ylabel(f'Amplitude {ylabel} (Pico)')
    plt.xlim(0, limite_freq_visual)
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.tight_layout()
    # [ATUALIZADO] Garante fundo branco ao salvar
    plt.savefig(pasta_saida / nome_arquivo_saida, format='svg', facecolor='white')
    plt.close()

def plotar_barras_harmonicas_v1v3(v1, v3, nome_variavel, caminho_salvar, label_y_unit="V"):
    percent = (v3/v1 * 100) if v1 > 0 else 0

    plt.figure()
    # [ATUALIZADO] Mudei edgecolor para 'black' (branco no fundo branco some)
    barras = plt.bar(['Fundamental (60Hz)', '3ª Harmônica (180Hz)'], [v1, v3],
                     color=['#1f77b4', '#d62728'], edgecolor='black', width=0.6)

    plt.ylabel(f'Magnitude ({label_y_unit})')
    titulo_limpo = nome_variavel.replace('_raw', '').replace('_', ' ')
    plt.title(f'{titulo_limpo}: Distorção 3ª Ordem\n(V3/V1 = {percent:.2f}%)')
    plt.grid(axis='y', linestyle='--', alpha=0.5)

    for rect in barras:
        height = rect.get_height()
        plt.text(rect.get_x() + rect.get_width()/2., height, f'{height:.1f} {label_y_unit}',
                 ha='center', va='bottom', fontsize=10)

    plt.tight_layout()
    plt.savefig(caminho_salvar, format='svg', facecolor='white')
    plt.close()

def plotar_envelope_rms_maximo(tempo, dados_tres_fases, caminho_salvar, label_y="Corrente"):
    imax_vetor = get_imax_envelope_vetor(tempo, dados_tres_fases)
    pico_max_absoluto = np.max(imax_vetor)

    plt.figure(figsize=(cm_to_inch(16), cm_to_inch(10)))
    plt.plot(tempo, imax_vetor, color='red', linewidth=1.5, label='Envelope Máximo')

    plt.xlabel('Tempo (s)')
    plt.ylabel(f'{label_y} Máxima (RMS)')
    plt.title(f'Envelope de {label_y} Máxima')

    # [ATUALIZADO] Caixa branca requer texto preto para leitura
    props = dict(boxstyle='round', facecolor='white', alpha=0.9, edgecolor='black')
    plt.text(0.5, 0.15, f'Max (RMS): {pico_max_absoluto:.1f}', transform=plt.gca().transAxes,
             fontsize=12, color='black', ha='center', bbox=props) # Cor do texto mudada para black

    plt.grid(True, linestyle='--', alpha=0.6)
    plt.tight_layout()
    plt.savefig(caminho_salvar, format='svg', facecolor='white')
    plt.close()

def plotar_rms_tres_fases_com_zoom(tempo, dados_tres_fases, nome_variavel, caminho_salvar, label_y="Corrente"):
    rms_a = calcular_rms_movel(dados_tres_fases[:, 0], tempo)
    rms_b = calcular_rms_movel(dados_tres_fases[:, 1], tempo)
    rms_c = calcular_rms_movel(dados_tres_fases[:, 2], tempo)

    imax_vetor = np.maximum.reduce([rms_a, rms_b, rms_c])
    idx_pico = np.argmax(imax_vetor)
    t_pico = tempo[idx_pico]
    valor_max_pico = imax_vetor[idx_pico]

    fig = plt.figure(figsize=(cm_to_inch(18), cm_to_inch(16)))
    gs = gridspec.GridSpec(2, 2, height_ratios=[2, 1])

    ax_main = fig.add_subplot(gs[0, :])
    ax_main.plot(tempo, rms_a, color='blue', label='Fase A', linewidth=1.2)
    ax_main.plot(tempo, rms_b, color='red', label='Fase B', linewidth=1.2)
    ax_main.plot(tempo, rms_c, color='green', label='Fase C', linewidth=1.2)
    ax_main.axvline(x=t_pico, color='red', linestyle='--', alpha=0.7)

    # [ATUALIZADO] Caixa com fundo branco e texto preto
    props = dict(boxstyle='round', facecolor='white', alpha=0.8, edgecolor='gray')
    ax_main.text(0.05, 0.95, f'Máx RMS: {valor_max_pico:.1f}', transform=ax_main.transAxes,
                 fontsize=11, verticalalignment='top', bbox=props, color='black')

    titulo_limpo = nome_variavel.replace('_raw', '').replace('_', ' ')
    ax_main.set_title(f'Análise RMS - {titulo_limpo}')
    ax_main.set_ylabel(f'{label_y} RMS')
    ax_main.legend(loc='upper right')
    ax_main.grid(True, linestyle=':', alpha=0.5)

    t_z1_ini, t_z1_fim = max(0, t_pico - 0.15), max(0, t_pico - 0.02)
    t_z2_ini, t_z2_fim = max(0, t_pico - 0.02), min(tempo[-1], t_pico + 0.10)

    ax_z1 = fig.add_subplot(gs[1, 0])
    ax_z1.plot(tempo, rms_a, 'b', tempo, rms_b, 'r', tempo, rms_c, 'g')
    ax_z1.set_xlim(t_z1_ini, t_z1_fim)
    mask1 = (tempo >= t_z1_ini) & (tempo <= t_z1_fim)
    if any(mask1):
        y_vals = np.concatenate([rms_a[mask1], rms_b[mask1], rms_c[mask1]])
        ax_z1.set_ylim(np.min(y_vals)*0.95, np.max(y_vals)*1.05)
    ax_z1.set_title('Zoom: Pré-Evento', fontsize=10)
    ax_z1.set_xlabel('Tempo (s)')
    ax_z1.set_ylabel(f'{label_y} RMS')
    ax_z1.grid(True, linestyle=':', alpha=0.5)

    ax_z2 = fig.add_subplot(gs[1, 1])
    ax_z2.plot(tempo, rms_a, 'b', tempo, rms_b, 'r', tempo, rms_c, 'g')
    ax_z2.set_xlim(t_z2_ini, t_z2_fim)
    ax_z2.axvline(x=t_pico, color='red', linestyle='--', alpha=0.7)
    mask2 = (tempo >= t_z2_ini) & (tempo <= t_z2_fim)
    if any(mask2):
         y_vals_2 = np.concatenate([rms_a[mask2], rms_b[mask2], rms_c[mask2]])
         ax_z2.set_ylim(bottom=0, top=max(valor_max_pico*1.1, np.max(y_vals_2)*1.05))
    ax_z2.set_title('Zoom: Evento Principal', fontsize=10)
    ax_z2.set_xlabel('Tempo (s)')
    ax_z2.grid(True, linestyle=':', alpha=0.5)

    plt.tight_layout()
    plt.savefig(caminho_salvar, format='svg', facecolor='white')
    plt.close()

def plotar_sequencias_simetricas(tempo, seq0, seq1, seq2, nome_variavel, caminho_salvar, label_y="Tensão"):
    plt.figure()

    plt.plot(tempo, seq1, color='blue', label='Positiva (1)', linewidth=1.5)
    plt.plot(tempo, seq2, color='red', label='Negativa (2)', linewidth=1.2, linestyle='--')
    plt.plot(tempo, seq0, color='green', label='Zero (0)', linewidth=1.2, linestyle=':')

    plt.xlabel('Tempo (s)')
    plt.ylabel(f'Magnitude {label_y} (RMS)')

    titulo_limpo = nome_variavel.replace('_raw', '').replace('_', ' ')
    plt.title(f'Componentes Simétricas - {titulo_limpo}')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.6)

    plt.tight_layout()
    plt.savefig(caminho_salvar, format='svg', facecolor='white')
    plt.close()

def plotar_comp_harmonicas_lado_a_lado(v1_a, v3_a, v1_b, v3_b, label_a, label_b, nome_barra, caminho_salvar):
    pct_a = (v3_a / v1_a * 100) if v1_a > 0 else 0
    pct_b = (v3_b / v1_b * 100) if v1_b > 0 else 0

    plt.figure()
    x_pos = np.arange(2)
    vals = [pct_a, pct_b]
    labels = [label_a, label_b]

    # [ATUALIZADO] Edgecolor black
    barras = plt.bar(x_pos, vals, color=['#1f77b4', '#d62728'], edgecolor='black', width=0.5)
    plt.xticks(x_pos, labels)
    plt.ylabel(r'Distorção $V_3 / V_1$ (%)')

    barra_limpa = nome_barra.replace('_raw', '').replace('V_', 'Barra ')
    plt.title(f'Comparação Harmônica (3ª Ordem) - {barra_limpa}')
    plt.grid(axis='y', linestyle='--', alpha=0.5)

    max_val = max(vals) if vals else 0
    if max_val > 0: plt.ylim(0, max_val * 1.25)

    for rect in barras:
        height = rect.get_height()
        plt.text(rect.get_x() + rect.get_width()/2., height * 1.02, f'{height:.2f}%',
                 ha='center', va='bottom', fontweight='bold')

    plt.tight_layout()
    plt.savefig(caminho_salvar, format='svg', facecolor='white')
    plt.close()

def plotar_comp_imax_envelope_tempo(t, imax_a, imax_b, label_a, label_b, nome_barra, caminho_salvar):
    plt.figure()
    plt.plot(t, imax_a, color='#1f77b4', label=label_a, linewidth=1.5)
    plt.plot(t, imax_b, color='#d62728', label=label_b, linewidth=1.5, linestyle='--')

    plt.xlabel('Tempo (s)')
    plt.ylabel('Corrente Máxima RMS (A)')

    barra_limpa = nome_barra.replace('_raw', '').replace('I_', 'Barra ')
    plt.title(f'Comparação de Envelope Máximo - {barra_limpa}')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.tight_layout()
    plt.savefig(caminho_salvar, format='svg', facecolor='white')
    plt.close()
## --- 1. CONFIGURAÇÕES GERAIS ---
pasta_raiz_resultados = Path('Resultados_Tese_2') # Onde as pastas serão criadas

# Sua lista de arquivos
lista_arquivos = [
    r'C:/Users/leosa/OneDrive/Coisas_Leonardo/gits/CurtosT2F/T2F_MATLAB/NovoArtigoPowerDelivery34bus/NovoModeloQualificacao/Teste_Novo_Sem_Terra_14/Processados_HDF5/MRT__Sem_Falta_py.mat',
    r'C:/Users/leosa/OneDrive/Coisas_Leonardo/gits/CurtosT2F/T2F_MATLAB/NovoArtigoPowerDelivery34bus/NovoModeloQualificacao/Teste_Novo_Sem_Terra_14/Processados_HDF5/MRT_SR__Sem_Falta_py.mat',
    r'C:/Users/leosa/OneDrive/Coisas_Leonardo/gits/CurtosT2F/T2F_MATLAB/NovoArtigoPowerDelivery34bus/NovoModeloQualificacao/Teste_Novo_Sem_Terra_14/Processados_HDF5/Qualificacao__Sem_Falta_py.mat',
    r'C:/Users/leosa/OneDrive/Coisas_Leonardo/gits/CurtosT2F/T2F_MATLAB/NovoArtigoPowerDelivery34bus/NovoModeloQualificacao/Teste_Novo_Sem_Terra_14/Processados_HDF5/Qualificacao_SR__Sem_Falta_py.mat',
    r'C:/Users/leosa/OneDrive/Coisas_Leonardo/gits/CurtosT2F/T2F_MATLAB/NovoArtigoPowerDelivery34bus/NovoModeloQualificacao/Teste_Novo_Sem_Terra_14/Processados_HDF5/MRT_sem_terra__Sem_Falta_py.mat',
    r'C:/Users/leosa/OneDrive/Coisas_Leonardo/gits/CurtosT2F/T2F_MATLAB/NovoArtigoPowerDelivery34bus/NovoModeloQualificacao/Teste_Novo_Sem_Terra_14/Processados_HDF5/MRT_SR_sem_terra__Sem_Falta_py.mat',
    r'C:/Users/leosa/OneDrive/Coisas_Leonardo/gits/CurtosT2F/T2F_MATLAB/NovoArtigoPowerDelivery34bus/NovoModeloQualificacao/Teste_Novo_Sem_Terra_14/Processados_HDF5/Qualificacao_sem_terra__Sem_Falta_py.mat',
    r'C:/Users/leosa/OneDrive/Coisas_Leonardo/gits/CurtosT2F/T2F_MATLAB/NovoArtigoPowerDelivery34bus/NovoModeloQualificacao/Teste_Novo_Sem_Terra_14/Processados_HDF5/MRT_sem_terra__A_822_-_Falta_A_py.mat',
    r'C:/Users/leosa/OneDrive/Coisas_Leonardo/gits/CurtosT2F/T2F_MATLAB/NovoArtigoPowerDelivery34bus/NovoModeloQualificacao/Teste_Novo_Sem_Terra_14/Processados_HDF5/MRT_SR__A_822_-_Falta_A_py.mat',
    r'C:/Users/leosa/OneDrive/Coisas_Leonardo/gits/CurtosT2F/T2F_MATLAB/NovoArtigoPowerDelivery34bus/NovoModeloQualificacao/Teste_Novo_Sem_Terra_14/Processados_HDF5/MRT_SR_sem_terra__A_822_-_Falta_A_py.mat',
    r'C:/Users/leosa/OneDrive/Coisas_Leonardo/gits/CurtosT2F/T2F_MATLAB/NovoArtigoPowerDelivery34bus/NovoModeloQualificacao/Teste_Novo_Sem_Terra_14/Processados_HDF5/Qualificacao__R_822_-_Falta_AB_py.mat',
    r'C:/Users/leosa/OneDrive/Coisas_Leonardo/gits/CurtosT2F/T2F_MATLAB/NovoArtigoPowerDelivery34bus/NovoModeloQualificacao/Teste_Novo_Sem_Terra_14/Processados_HDF5/Qualificacao_sem_terra__R_822_-_Falta_AB_py.mat',
    r'C:/Users/leosa/OneDrive/Coisas_Leonardo/gits/CurtosT2F/T2F_MATLAB/NovoArtigoPowerDelivery34bus/NovoModeloQualificacao/Teste_Novo_Sem_Terra_14/Processados_HDF5/Qualificacao_SR__R_822_-_Falta_AB_py.mat',
    r'C:/Users/leosa/OneDrive/Coisas_Leonardo/gits/CurtosT2F/T2F_MATLAB/NovoArtigoPowerDelivery34bus/NovoModeloQualificacao/Teste_Novo_Sem_Terra_14/Processados_HDF5/Qualificacao_SR_sem_terra__R_822_-_Falta_AB_py.mat',
    r'C:/Users/leosa/OneDrive/Coisas_Leonardo/gits/CurtosT2F/T2F_MATLAB/NovoArtigoPowerDelivery34bus/NovoModeloQualificacao/Teste_Novo_Sem_Terra_14/Processados_HDF5/Qualificacao_SR_sem_terra__Sem_Falta_py.mat',
]

print(f"Iniciando processamento unificado de {len(lista_arquivos)} arquivos...\n")

for caminho_str in lista_arquivos:
    caminho_arquivo = Path(caminho_str)

    try:
        # 1. Cria a pasta de saída para este arquivo
        pasta_final = pasta_raiz_resultados / caminho_arquivo.stem
        pasta_final.mkdir(parents=True, exist_ok=True)

        print(f"--> Processando: {caminho_arquivo.name}")

        # 2. Abre o arquivo HDF5
        with h5py.File(caminho_arquivo, 'r') as f:
            t = np.array(f['t']).flatten()

            # Lista de todas as variáveis
            todas_vars = [
                ('V_800_raw', 'Tensão (V)', 'V_barra_800'),
                ('V_T2F_raw', 'Tensão (V)', 'V_barra_T2F'),
                ('V_T2F1_raw', 'Tensão (V)', 'V_barra_T2F1'),
                ('V_818_raw', 'Tensão (V)', 'V_barra_818'),
                ('V_820_raw', 'Tensão (V)', 'V_barra_820'),
                ('V_822_raw', 'Tensão (V)', 'V_barra_822'),
                ('I_800_raw', 'Corrente (A)', 'I_barra_800'),
                ('I_T2F_raw', 'Corrente (A)', 'I_barra_T2F'),
                ('I_T2F1_raw', 'Corrente (A)', 'I_barra_T2F1'),
                ('I_818_raw', 'Corrente (A)', 'I_barra_818'),
                ('I_820_raw', 'Corrente (A)', 'I_barra_820'),
                ('I_822_raw', 'Corrente (A)', 'I_barra_822'),
            ]

            for var_mat, label_y_plot, sufixo_nome in todas_vars:
                if var_mat in f:
                    # ==========================================================
                    # PADRONIZAÇÃO DE DADOS
                    # ==========================================================
                    dados_raw = np.array(f[var_mat])

                    if dados_raw.ndim == 2:
                        if dados_raw.shape[0] < dados_raw.shape[1]:
                            dados_raw = dados_raw.T
                    elif dados_raw.ndim == 1:
                        dados_raw = dados_raw.reshape(-1, 1)

                    linhas, colunas = dados_raw.shape
                    dados_padronizados = np.zeros((linhas, 3))

                    for c in range(min(colunas, 3)):
                        dados_padronizados[:, c] = dados_raw[:, c]

                    dados = dados_padronizados
                    # ==========================================================

                    # Só processa se houver dados não nulos
                    if np.max(np.abs(dados)) > 1e-9:

                        # --- TAREFA A: FFT ---
                        nome_fft = f"fft_{sufixo_nome}.svg"
                        gerar_grafico_fft_espectro(t, dados, nome_fft, pasta_final, label_y_plot, limite_freq_visual=1000)

                        # --- TAREFA B: BARRAS V1/V3 ---
                        if var_mat.startswith('V'):
                            nome_barras = f"harmonicas_{sufixo_nome}.svg"
                            v1, v3 = calcular_v1_v3_global(dados[:, 0], t, FREQUENCIA_REDE)
                            plotar_barras_harmonicas_v1v3(v1, v3, var_mat, pasta_final / nome_barras, label_y_unit="V")

                        # --- TAREFA C: RMS DETALHADO ---
                        nome_env = f"rms_envelope_{sufixo_nome}.svg"
                        plotar_envelope_rms_maximo(t, dados, pasta_final / nome_env, label_y=label_y_plot.split(' ')[0])

                        nome_zoom = f"rms_zoom_{sufixo_nome}.svg"
                        plotar_rms_tres_fases_com_zoom(t, dados, var_mat, pasta_final / nome_zoom, label_y=label_y_plot.split(' ')[0])

                        # ======================================================
                        # TAREFA D: SEQUÊNCIAS (Zero, Positiva, Negativa)
                        # ======================================================
                        # 1. Calcula as sequências
                        seq0, seq1, seq2 = calcular_componentes_simetricas_tempo(dados, t, FREQUENCIA_REDE)

                        # 2. Gera o gráfico
                        nome_seq = f"sequencias_{sufixo_nome}.svg"
                        label_tipo = "Tensão" if "V_" in var_mat else "Corrente"
                        plotar_sequencias_simetricas(t, seq0, seq1, seq2, var_mat, pasta_final / nome_seq, label_y=label_tipo)
                        # ======================================================

        print("    -> OK.")

    except Exception as e:
        print(f"ERRO CRÍTICO ao processar {caminho_arquivo.name}:\n{e}\n")

print("\n--- Processamento Unificado Finalizado ---")

In [4]:
# --- 1. CONFIGURAÇÕES ---
FREQ_SISTEMA = 60

# LISTA DE PARES PARA COMPARAR (Arquivo 1, Arquivo 2)
# Usei raw strings (r'...') para evitar problemas com as barras do Windows
# [ATUALIZADO] Configurações para forçar fundo branco
# [ATUALIZADO] Configurações para fundo branco E TEXTO PRETO
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman"],
    "font.size": 12,
    "figure.figsize": (cm_to_inch(16), cm_to_inch(10)),

    # Cores de Fundo
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
    "savefig.transparent": False,

    # Cores de Texto e Linhas (Tudo Preto)
    "text.color": "black",
    "axes.labelcolor": "black",
    "xtick.color": "black",
    "ytick.color": "black",
    "axes.edgecolor": "black"
})

# --- 2. FUNÇÕES MATEMÁTICAS (Incluindo Componentes Simétricas e TCC) ---

def calcular_rms_movel(sinal, tempo, freq_rede=60):
    if len(tempo) < 2: return np.zeros_like(sinal)
    dt = tempo[1] - tempo[0]
    if dt <= 0: return np.zeros_like(sinal)
    fs = 1 / dt
    janela = int(fs / freq_rede)
    if janela < 1: janela = 1
    sinal_quadrado = sinal ** 2
    janela_media = np.ones(janela) / janela
    return np.sqrt(np.convolve(sinal_quadrado, janela_media, mode='same'))

def extrair_fasor_dinamico(sinal, tempo, freq=60):
    """Extrai fasor complexo (Mag e Angulo) no tempo via DFT."""
    dt = tempo[1] - tempo[0]
    if dt <= 0: return np.zeros_like(sinal, dtype=complex)
    samples = int((1/freq)/dt)
    if samples < 1: samples = 1
    t_win = np.arange(samples) * dt
    k_cos = np.cos(2*np.pi*freq*t_win) * (2/samples)
    k_sin = np.sin(2*np.pi*freq*t_win) * (2/samples)
    real = np.convolve(sinal, k_cos, mode='same')
    imag = np.convolve(sinal, k_sin, mode='same')
    return real - 1j*imag

def calcular_sequencias(dados_3fases, tempo, freq=60):
    """Retorna I0, I1, I2 (Magnitudes) ao longo do tempo."""
    Va = extrair_fasor_dinamico(dados_3fases[:, 0], tempo, freq)
    Vb = extrair_fasor_dinamico(dados_3fases[:, 1], tempo, freq)
    Vc = extrair_fasor_dinamico(dados_3fases[:, 2], tempo, freq)
    a = np.exp(1j * 2 * np.pi / 3)

    # Fortescue
    V0 = (Va + Vb + Vc) / 3
    V1 = (Va + a*Vb + a**2*Vc) / 3
    V2 = (Va + a**2*Vb + a*Vc) / 3

    return np.abs(V0), np.abs(V1), np.abs(V2)

def calcular_tempo_tcc_iec(corrente, i_pickup=10, time_dial=0.5, curve_type='VI'):
    """
    Calcula tempo de atuação baseado na norma IEC 60255.
    curve_type: 'SI' (Standard Inverse), 'VI' (Very Inverse), 'EI' (Extremely Inverse)
    """
    # Parâmetros IEC (k, alpha)
    curves = {
        'SI': (0.14, 0.02),
        'VI': (13.5, 1.0),
        'EI': (80.0, 2.0)
    }
    k, alpha = curves.get(curve_type, (13.5, 1.0))

    if corrente < i_pickup:
        return None # Não atua

    M = corrente / i_pickup
    t = time_dial * (k / (M**alpha - 1))
    return t

def ler_dados_padronizados(f_obj, var_nome):
    if var_nome not in f_obj: return None
    dados_raw = np.array(f_obj[var_nome])
    if dados_raw.ndim == 2:
        if dados_raw.shape[0] < dados_raw.shape[1]: dados_raw = dados_raw.T
    elif dados_raw.ndim == 1: dados_raw = dados_raw.reshape(-1, 1)

    linhas, colunas = dados_raw.shape
    dados_pad = np.zeros((linhas, 3))
    for c in range(min(colunas, 3)):
        dados_pad[:, c] = dados_raw[:, c]
    return dados_pad

def get_imax_envelope_vetor(tempo, dados):
    rms_a = calcular_rms_movel(dados[:, 0], tempo)
    rms_b = calcular_rms_movel(dados[:, 1], tempo)
    rms_c = calcular_rms_movel(dados[:, 2], tempo)
    return np.maximum.reduce([rms_a, rms_b, rms_c])

# --- 3. FUNÇÕES DE PLOTAGEM AVANÇADAS ---

def plotar_comparacao_seq_zero(t, i0_a, i0_b, lab_a, lab_b, nome_barra, pasta):
    """Plota a Sequência Zero (I0) dos dois casos."""
    plt.figure()
    plt.plot(t, i0_a, color='black', label=f'{lab_a} (I0)', linewidth=1.5)
    plt.plot(t, i0_b, color='red', label=f'{lab_b} (I0)', linewidth=1.5, linestyle='--')
    plt.xlabel('Tempo (s)'); plt.ylabel('Corrente de Sequência Zero (A)')
    plt.title(f'Comparação I0 - {nome_barra.replace("_raw","")}')
    plt.legend(); plt.grid(True, linestyle='--', alpha=0.6)
    plt.tight_layout()
    plt.savefig(pasta / f"SeqZero_{nome_barra}.svg", format='svg', facecolor='white')
    plt.close()

def plotar_tcc_coordenacao(imax_a, imax_b, lab_a, lab_b, nome_barra, pasta):
    """
    Gera curva TCC e plota o ponto de operação máximo de cada caso.
    """
    # Configuração do Relé (Exemplo para Tese)
    I_PICKUP = 50 # Amperes
    TIME_DIAL = 0.5
    TIPO_CURVA = 'VI' # Very Inverse

    # Gera curva teórica
    correntes_x = np.logspace(np.log10(I_PICKUP*1.1), np.log10(max(imax_a, imax_b)*1.5), 100)
    tempos_y = [calcular_tempo_tcc_iec(i, I_PICKUP, TIME_DIAL, TIPO_CURVA) for i in correntes_x]

    plt.figure()
    plt.loglog(correntes_x, tempos_y, 'k-', label=f'Curva IEC {TIPO_CURVA} (TD={TIME_DIAL})')

    # Ponto A
    t_a = calcular_tempo_tcc_iec(imax_a, I_PICKUP, TIME_DIAL, TIPO_CURVA)
    if t_a:
        plt.plot(imax_a, t_a, 'bo', markersize=8, label=f'{lab_a}: {imax_a:.0f}A ({t_a:.2f}s)')

    # Ponto B
    t_b = calcular_tempo_tcc_iec(imax_b, I_PICKUP, TIME_DIAL, TIPO_CURVA)
    if t_b:
        plt.plot(imax_b, t_b, 'rx', markersize=8, markeredgewidth=2, label=f'{lab_b}: {imax_b:.0f}A ({t_b:.2f}s)')

    plt.xlabel('Corrente (A)'); plt.ylabel('Tempo de Atuação (s)')
    plt.title(f'Coordenação TCC - {nome_barra.replace("_raw","")}')
    plt.grid(True, which="both", ls="--", alpha=0.4)
    plt.legend()
    plt.tight_layout()
    plt.savefig(pasta / f"TCC_{nome_barra}.svg", format='svg', facecolor='white')
    plt.close()

def plotar_perfil_tensao_regime(val_a, val_b, lab_a, lab_b, nome_barra, pasta):
    """Comparativo de Tensão em Regime (Pré-Falta)."""
    plt.figure()
    plt.bar(['Caso A', 'Caso B'], [val_a, val_b], color=['blue', 'red'], edgecolor='black', width=0.5)
    plt.ylabel('Tensão Pré-Falta (V)')
    plt.title(f'Perfil de Tensão - {nome_barra.replace("_raw","")}')
    plt.ylim(0, max(val_a, val_b)*1.2)

    for i, v in enumerate([val_a, val_b]):
        plt.text(i, v*1.01, f'{v:.1f} V', ha='center', color='black', fontweight='bold')

    plt.tight_layout()
    plt.savefig(caminho_salvar, format='svg', facecolor='white') # Corrigido uso da variavel pasta
    plt.close()

# --- 4. LOOP PRINCIPAL ---

lista_de_comparacoes = [
    (
        r'C:/Users/leosa/OneDrive/Coisas_Leonardo/gits/CurtosT2F/T2F_MATLAB/NovoArtigoPowerDelivery34bus/NovoModeloQualificacao/Teste_Novo_Sem_Terra_14/Processados_HDF5/MRT__Sem_Falta_py.mat',
        r'C:/Users/leosa/OneDrive/Coisas_Leonardo/gits/CurtosT2F/T2F_MATLAB/NovoArtigoPowerDelivery34bus/NovoModeloQualificacao/Teste_Novo_Sem_Terra_14/Processados_HDF5/MRT_SR__Sem_Falta_py.mat'
    ),
    (
        r'C:/Users/leosa/OneDrive/Coisas_Leonardo/gits/CurtosT2F/T2F_MATLAB/NovoArtigoPowerDelivery34bus/NovoModeloQualificacao/Teste_Novo_Sem_Terra_14/Processados_HDF5/Qualificacao__Sem_Falta_py.mat',
        r'C:/Users/leosa/OneDrive/Coisas_Leonardo/gits/CurtosT2F/T2F_MATLAB/NovoArtigoPowerDelivery34bus/NovoModeloQualificacao/Teste_Novo_Sem_Terra_14/Processados_HDF5/Qualificacao_SR__Sem_Falta_py.mat'
    ),
    (
        r'C:/Users/leosa/OneDrive/Coisas_Leonardo/gits/CurtosT2F/T2F_MATLAB/NovoArtigoPowerDelivery34bus/NovoModeloQualificacao/Teste_Novo_Sem_Terra_14/Processados_HDF5/MRT_sem_terra__Sem_Falta_py.mat',
        r'C:/Users/leosa/OneDrive/Coisas_Leonardo/gits/CurtosT2F/T2F_MATLAB/NovoArtigoPowerDelivery34bus/NovoModeloQualificacao/Teste_Novo_Sem_Terra_14/Processados_HDF5/MRT_SR_sem_terra__Sem_Falta_py.mat'
    ),
    (
        r'C:/Users/leosa/OneDrive/Coisas_Leonardo/gits/CurtosT2F/T2F_MATLAB/NovoArtigoPowerDelivery34bus/NovoModeloQualificacao/Teste_Novo_Sem_Terra_14/Processados_HDF5/Qualificacao_sem_terra__Sem_Falta_py.mat',
        r'C:/Users/leosa/OneDrive/Coisas_Leonardo/gits/CurtosT2F/T2F_MATLAB/NovoArtigoPowerDelivery34bus/NovoModeloQualificacao/Teste_Novo_Sem_Terra_14/Processados_HDF5/Qualificacao_SR_sem_terra__Sem_Falta_py.mat'
    ),
    (
        r'C:/Users/leosa/OneDrive/Coisas_Leonardo/gits/CurtosT2F/T2F_MATLAB/NovoArtigoPowerDelivery34bus/NovoModeloQualificacao/Teste_Novo_Sem_Terra_14/Processados_HDF5/MRT__Sem_Falta_py.mat',
        r'C:/Users/leosa/OneDrive/Coisas_Leonardo/gits/CurtosT2F/T2F_MATLAB/NovoArtigoPowerDelivery34bus/NovoModeloQualificacao/Teste_Novo_Sem_Terra_14/Processados_HDF5/Qualificacao__Sem_Falta_py.mat'
    ),
    (
        r'C:/Users/leosa/OneDrive/Coisas_Leonardo/gits/CurtosT2F/T2F_MATLAB/NovoArtigoPowerDelivery34bus/NovoModeloQualificacao/Teste_Novo_Sem_Terra_14/Processados_HDF5/MRT_SR__Sem_Falta_py.mat',
        r'C:/Users/leosa/OneDrive/Coisas_Leonardo/gits/CurtosT2F/T2F_MATLAB/NovoArtigoPowerDelivery34bus/NovoModeloQualificacao/Teste_Novo_Sem_Terra_14/Processados_HDF5/Qualificacao_SR__Sem_Falta_py.mat'
    ),
    (
        r'C:/Users/leosa/OneDrive/Coisas_Leonardo/gits/CurtosT2F/T2F_MATLAB/NovoArtigoPowerDelivery34bus/NovoModeloQualificacao/Teste_Novo_Sem_Terra_14/Processados_HDF5/MRT_sem_terra__Sem_Falta_py.mat',
        r'C:/Users/leosa/OneDrive/Coisas_Leonardo/gits/CurtosT2F/T2F_MATLAB/NovoArtigoPowerDelivery34bus/NovoModeloQualificacao/Teste_Novo_Sem_Terra_14/Processados_HDF5/Qualificacao_sem_terra__Sem_Falta_py.mat'
    ),
    (
        r'C:/Users/leosa/OneDrive/Coisas_Leonardo/gits/CurtosT2F/T2F_MATLAB/NovoArtigoPowerDelivery34bus/NovoModeloQualificacao/Teste_Novo_Sem_Terra_14/Processados_HDF5/MRT_SR_sem_terra__Sem_Falta_py.mat',
        r'C:/Users/leosa/OneDrive/Coisas_Leonardo/gits/CurtosT2F/T2F_MATLAB/NovoArtigoPowerDelivery34bus/NovoModeloQualificacao/Teste_Novo_Sem_Terra_14/Processados_HDF5/Qualificacao_SR_sem_terra__Sem_Falta_py.mat'
    )

]
# Lista para armazenar dados da tabela
tabela_indicadores = []

if lista_de_comparacoes:
    print(f"\nIniciando processamento avançado de {len(lista_de_comparacoes)} pares...\n")

    for i, (path1_str, path2_str) in enumerate(lista_de_comparacoes):
        arq1, arq2 = Path(path1_str), Path(path2_str)

        try:
            print(f"[{i+1}] Processando: {arq1.stem} vs {arq2.stem}")

            # Legendas
            l1, l2 = "Caso A", "Caso B"
            if "_SR_" in arq1.stem: l1 = "Sem Reg"
            elif "MRT" in arq1.stem: l1 = "MRT"
            if "_SR_" in arq2.stem: l2 = "Sem Reg"
            elif "Qualificacao" in arq2.stem: l2 = "T2F"
            if l1 == l2: l2 = l2 + " (Comp)"

            with h5py.File(arq1, 'r') as f1, h5py.File(arq2, 'r') as f2:
                t = np.array(f1['t']).flatten()

                # Variáveis de Interesse
                vars_corr = ['I_800_raw', 'I_T2F_raw', 'I_822_raw']
                vars_tens = ['V_800_raw', 'V_T2F_raw', 'V_822_raw']

                # --- ANÁLISE DE CORRENTE (I0, TCC, Máximos) ---
                for var in vars_corr:
                    d1 = ler_dados_padronizados(f1, var)
                    d2 = ler_dados_padronizados(f2, var)

                    if d1 is not None and d2 is not None:
                        # 1. Componentes Simétricas
                        i0_a, i1_a, i2_a = calcular_sequencias(d1, t)
                        i0_b, i1_b, i2_b = calcular_sequencias(d2, t)

                        # 2. Plotar Comparação I0 (Destaque da sua tese)
                        plotar_comparacao_seq_zero(t, i0_a, i0_b, l1, l2, var, pasta_raiz_resultados)

                        # 3. Calcular Envelope Máximo para TCC
                        imax_a = np.max(get_imax_envelope_vetor(t, d1))
                        imax_b = np.max(get_imax_envelope_vetor(t, d2))

                        # 4. Plotar TCC
                        plotar_tcc_coordenacao(imax_a, imax_b, l1, l2, var, pasta_raiz_resultados)

                        # 5. Salvar na Tabela (Indicadores Agregados)
                        # Calcula Desequilíbrio Máximo (I2/I1)
                        # Evita divisão por zero
                        with np.errstate(divide='ignore', invalid='ignore'):
                            unb_a = np.max(np.nan_to_num(i2_a / i1_a)) * 100
                            unb_b = np.max(np.nan_to_num(i2_b / i1_b)) * 100

                        tabela_indicadores.append({
                            'Par': i+1, 'Barra': var, 'Tipo': 'Corrente',
                            'Caso A': l1, 'Imax_A': imax_a, 'I0_Max_A': np.max(i0_a), 'Unbalance_A(%)': unb_a,
                            'Caso B': l2, 'Imax_B': imax_b, 'I0_Max_B': np.max(i0_b), 'Unbalance_B(%)': unb_b
                        })

                # --- ANÁLISE DE TENSÃO (Regime, VUF) ---
                for var in vars_tens:
                    d1 = ler_dados_padronizados(f1, var)
                    d2 = ler_dados_padronizados(f2, var)

                    if d1 is not None and d2 is not None:
                        # 1. Tensão de Regime (Média dos primeiros 0.05s)
                        idx_pre = int(0.05 / (t[1]-t[0]))
                        rms_a_regime = np.mean(calcular_rms_movel(d1[:idx_pre, 0], t[:idx_pre]))
                        rms_b_regime = np.mean(calcular_rms_movel(d2[:idx_pre, 0], t[:idx_pre]))

                        # Plota Perfil
                        plt.figure()
                        plt.bar([l1, l2], [rms_a_regime, rms_b_regime], color=['blue', 'red'], edgecolor='black')
                        plt.title(f'Tensão Regime - {var}'); plt.ylabel('V')
                        plt.savefig(pasta_raiz_resultados / f"PerfilTensao_{var}.svg", facecolor='white')
                        plt.close()

                        # 2. Componentes e VUF
                        v0_a, v1_a, v2_a = calcular_sequencias(d1, t)
                        v0_b, v1_b, v2_b = calcular_sequencias(d2, t)

                        with np.errstate(divide='ignore', invalid='ignore'):
                            vuf_a = np.max(np.nan_to_num(v2_a / v1_a)) * 100
                            vuf_b = np.max(np.nan_to_num(v2_b / v1_b)) * 100

                        tabela_indicadores.append({
                            'Par': i+1, 'Barra': var, 'Tipo': 'Tensao',
                            'Caso A': l1, 'V_Regime_A': rms_a_regime, 'VUF_Max_A(%)': vuf_a,
                            'Caso B': l2, 'V_Regime_B': rms_b_regime, 'VUF_Max_B(%)': vuf_b
                        })

        except Exception as e:
            print(f"ERRO no par {i+1}: {e}")

    # --- SALVAR TABELA FINAL (CSV) ---
    if tabela_indicadores:
        df = pd.DataFrame(tabela_indicadores)
        caminho_csv = pasta_raiz_resultados / "Tabela_Indicadores_Tese.csv"
        df.to_csv(caminho_csv, index=False, sep=';', decimal=',')
        print(f"\n--> Tabela de indicadores salva em: {caminho_csv}")
        print("--> Dica: Abra o CSV no Excel usando 'Dados > Texto para Colunas' se necessário.")

    print("\n--- Processamento Completo Finalizado ---")


Iniciando processamento avançado de 8 pares...

[1] Processando: MRT__Sem_Falta_py vs MRT_SR__Sem_Falta_py
[2] Processando: Qualificacao__Sem_Falta_py vs Qualificacao_SR__Sem_Falta_py
[3] Processando: MRT_sem_terra__Sem_Falta_py vs MRT_SR_sem_terra__Sem_Falta_py
[4] Processando: Qualificacao_sem_terra__Sem_Falta_py vs Qualificacao_SR_sem_terra__Sem_Falta_py
[5] Processando: MRT__Sem_Falta_py vs Qualificacao__Sem_Falta_py
[6] Processando: MRT_SR__Sem_Falta_py vs Qualificacao_SR__Sem_Falta_py
[7] Processando: MRT_sem_terra__Sem_Falta_py vs Qualificacao_sem_terra__Sem_Falta_py
[8] Processando: MRT_SR_sem_terra__Sem_Falta_py vs Qualificacao_SR_sem_terra__Sem_Falta_py

--> Tabela de indicadores salva em: Resultados_Tese_2\Tabela_Indicadores_Tese.csv
--> Dica: Abra o CSV no Excel usando 'Dados > Texto para Colunas' se necessário.

--- Processamento Completo Finalizado ---


In [11]:
# -*- coding: utf-8 -*-
import numpy as np
import matplotlib.pyplot as plt
import h5py
import pandas as pd
from pathlib import Path

# -------------------------------------------------------------------
# 0. FUNÇÕES BÁSICAS / CONFIGURAÇÕES
# -------------------------------------------------------------------

def cm_to_inch(*dims_cm):
    """Converte dimensões em cm para polegadas."""
    return tuple(d / 2.54 for d in dims_cm)

# Pasta raiz para salvar resultados – ajuste conforme necessário
pasta_raiz_resultados = Path(
    r'C:/GarotoDePrograma/Gits/NovoModeloQualificacao_Streamlit/Resultados_Tese_2'
)
pasta_raiz_resultados.mkdir(parents=True, exist_ok=True)

# --- 1. CONFIGURAÇÕES ---
FREQ_SISTEMA = 60

# Configurações Matplotlib: fundo branco e texto preto
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman"],
    "font.size": 12,
    "figure.figsize": cm_to_inch(16, 10),
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
    "savefig.transparent": False,
    "text.color": "black",
    "axes.labelcolor": "black",
    "xtick.color": "black",
    "ytick.color": "black",
    "axes.edgecolor": "black"
})

# -------------------------------------------------------------------
# 1.A – FUNÇÃO PARA "TRADUZIR" O NOME DO ARQUIVO
# -------------------------------------------------------------------

def descrever_arquivo(stem: str) -> str:
    """
    Recebe o nome-base do arquivo (sem extensão) e devolve uma descrição legível.
    Interpreta:
      - MRT / Qualificacao  -> topologia
      - _SR_                -> sem regulador
      - sem_terra           -> sem aterramento
      - __Sem_Falta_py      -> caso de regime (sem falta)
    """
    partes = []

    # Topologia
    if "MRT" in stem:
        partes.append("MRT (Monofásico c/ Retorno por Terra/Neutro)")
    elif "Qualificacao" in stem:
        partes.append("T2F (Trifásico a Dois Fios)")
    else:
        partes.append("Topologia não identificada")

    # Regulador
    if "_SR_" in stem:
        partes.append("Sem regulador")
    else:
        partes.append("Com regulador")

    # Aterramento
    if "sem_terra" in stem:
        partes.append("Sem aterramento da subestação")
    else:
        partes.append("Com aterramento da subestação")

    # Falta
    if "Sem_Falta" in stem:
        partes.append("Caso em regime (sem falta)")
    else:
        partes.append("Caso com falta (não explicitado)")

    return " | ".join(partes)

# -------------------------------------------------------------------
# 2. FUNÇÕES MATEMÁTICAS (RMS, FASORES, COMPONENTES, TCC)
# -------------------------------------------------------------------

def calcular_rms_movel(sinal, tempo, freq_rede=60):
    if len(tempo) < 2:
        return np.zeros_like(sinal)
    dt = tempo[1] - tempo[0]
    if dt <= 0:
        return np.zeros_like(sinal)
    fs = 1.0 / dt
    janela = int(fs / freq_rede)
    if janela < 1:
        janela = 1
    sinal_quadrado = sinal ** 2
    janela_media = np.ones(janela) / janela
    return np.sqrt(np.convolve(sinal_quadrado, janela_media, mode='same'))

def extrair_fasor_dinamico(sinal, tempo, freq=60):
    if len(tempo) < 2:
        return np.zeros_like(sinal, dtype=complex)
    dt = tempo[1] - tempo[0]
    if dt <= 0:
        return np.zeros_like(sinal, dtype=complex)

    samples = int((1.0/freq) / dt)
    if samples < 1:
        samples = 1

    t_win = np.arange(samples) * dt
    k_cos = np.cos(2*np.pi*freq*t_win) * (2.0/samples)
    k_sin = np.sin(2*np.pi*freq*t_win) * (2.0/samples)

    real = np.convolve(sinal, k_cos, mode='same')
    imag = np.convolve(sinal, k_sin, mode='same')
    return real - 1j*imag

def calcular_sequencias(dados_3fases, tempo, freq=60):
    Va = extrair_fasor_dinamico(dados_3fases[:, 0], tempo, freq)
    Vb = extrair_fasor_dinamico(dados_3fases[:, 1], tempo, freq)
    Vc = extrair_fasor_dinamico(dados_3fases[:, 2], tempo, freq)

    a = np.exp(1j * 2 * np.pi / 3)

    V0 = (Va + Vb + Vc) / 3.0
    V1 = (Va + a*Vb + a**2 * Vc) / 3.0
    V2 = (Va + a**2 * Vb + a * Vc) / 3.0

    return np.abs(V0), np.abs(V1), np.abs(V2)

def calcular_tempo_tcc_iec(corrente, i_pickup=10, time_dial=0.5, curve_type='VI'):
    curves = {
        'SI': (0.14, 0.02),
        'VI': (13.5, 1.0),
        'EI': (80.0, 2.0)
    }
    k, alpha = curves.get(curve_type, (13.5, 1.0))

    if corrente <= 0 or corrente < i_pickup:
        return None

    M = corrente / i_pickup
    if M <= 1.0:
        return None

    t = time_dial * (k / (M**alpha - 1.0))
    return t

def ler_dados_padronizados(f_obj, var_nome):
    if var_nome not in f_obj:
        return None
    dados_raw = np.array(f_obj[var_nome])

    if dados_raw.ndim == 2:
        if dados_raw.shape[0] < dados_raw.shape[1]:
            dados_raw = dados_raw.T
    elif dados_raw.ndim == 1:
        dados_raw = dados_raw.reshape(-1, 1)

    linhas, colunas = dados_raw.shape
    dados_pad = np.zeros((linhas, 3))
    for c in range(min(colunas, 3)):
        dados_pad[:, c] = dados_raw[:, c]
    return dados_pad

def get_imax_envelope_vetor(tempo, dados):
    rms_a = calcular_rms_movel(dados[:, 0], tempo)
    rms_b = calcular_rms_movel(dados[:, 1], tempo)
    rms_c = calcular_rms_movel(dados[:, 2], tempo)
    return np.maximum.reduce([rms_a, rms_b, rms_c])

# -------------------------------------------------------------------
# 3. FUNÇÕES DE PLOTAGEM (AGORA COM ID DO PAR)
# -------------------------------------------------------------------

def plotar_comparacao_seq_zero(t, i0_a, i0_b, lab_a, lab_b, nome_barra, pasta, par_id):
    plt.figure()
    plt.plot(t, i0_a, color='black', label=f'{lab_a} (I0)', linewidth=1.5)
    plt.plot(t, i0_b, color='red', label=f'{lab_b} (I0)', linewidth=1.5, linestyle='--')
    plt.xlabel('Tempo (s)')
    plt.ylabel('Corrente de Sequência Zero (A)')
    plt.title(f'Comparação I0 - {nome_barra.replace("_raw","")}')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.tight_layout()
    # INCLUIR O ID DO PAR NO NOME DO ARQUIVO
    plt.savefig(pasta / f"Par{par_id:02d}_SeqZero_{nome_barra}.svg", format='svg', facecolor='white')
    plt.close()

def plotar_tcc_coordenacao(imax_a, imax_b, lab_a, lab_b, nome_barra, pasta, par_id):
    I_PICKUP = 50.0
    TIME_DIAL = 0.5
    TIPO_CURVA = 'VI'

    imax_global = max(imax_a, imax_b)
    if imax_global <= I_PICKUP * 1.1:
        return

    correntes_x = np.logspace(
        np.log10(I_PICKUP*1.1),
        np.log10(imax_global*1.5),
        200
    )
    tempos_y = []
    for i in correntes_x:
        t_val = calcular_tempo_tcc_iec(i, I_PICKUP, TIME_DIAL, TIPO_CURVA)
        if t_val is None or t_val <= 0:
            t_val = np.nan
        tempos_y.append(t_val)
    tempos_y = np.array(tempos_y)

    plt.figure()
    plt.loglog(correntes_x, tempos_y, 'k-', label=f'IEC {TIPO_CURVA} (TD={TIME_DIAL})')

    t_a = calcular_tempo_tcc_iec(imax_a, I_PICKUP, TIME_DIAL, TIPO_CURVA)
    if t_a is not None and t_a > 0:
        plt.loglog(imax_a, t_a, 'bo', markersize=8,
                   label=f'{lab_a}: {imax_a:.0f} A / {t_a:.2f} s')

    t_b = calcular_tempo_tcc_iec(imax_b, I_PICKUP, TIME_DIAL, TIPO_CURVA)
    if t_b is not None and t_b > 0:
        plt.loglog(imax_b, t_b, 'rx', markersize=8, markeredgewidth=2,
                   label=f'{lab_b}: {imax_b:.0f} A / {t_b:.2f} s')

    plt.xlabel('Corrente (A)')
    plt.ylabel('Tempo de Atuação (s)')
    plt.title(f'Coordenação TCC - {nome_barra.replace("_raw","")}')
    plt.grid(True, which="both", ls="--", alpha=0.4)
    plt.legend()
    plt.tight_layout()
    # INCLUIR O ID DO PAR NO NOME DO ARQUIVO
    plt.savefig(pasta / f"Par{par_id:02d}_TCC_{nome_barra}.svg", format='svg', facecolor='white')
    plt.close()

def plotar_perfil_tensao_regime(val_a, val_b, lab_a, lab_b, nome_barra, pasta, par_id):
    plt.figure()
    plt.bar([lab_a, lab_b], [val_a, val_b],
            color=['blue', 'red'], edgecolor='black', width=0.5)
    plt.ylabel('Tensão Pré-Falta (V)')
    plt.title(f'Perfil de Tensão - {nome_barra.replace("_raw","")}')
    plt.ylim(0, max(val_a, val_b)*1.2 if max(val_a, val_b) > 0 else 1)

    for i, v in enumerate([val_a, val_b]):
        plt.text(i, v*1.01, f'{v:.1f} V', ha='center',
                 color='black', fontweight='bold')

    plt.tight_layout()
    # INCLUIR O ID DO PAR NO NOME DO ARQUIVO
    plt.savefig(pasta / f"Par{par_id:02d}_PerfilTensao_{nome_barra}.svg",
                format='svg', facecolor='white')
    plt.close()

# -------------------------------------------------------------------
# 4. LOOP PRINCIPAL
# -------------------------------------------------------------------

lista_de_comparacoes = [
    (
        r'C:/Users/leosa/OneDrive/Coisas_Leonardo/gits/CurtosT2F/T2F_MATLAB/NovoArtigoPowerDelivery34bus/NovoModeloQualificacao/Teste_Novo_Sem_Terra_14/Processados_HDF5/MRT__Sem_Falta_py.mat',
        r'C:/Users/leosa/OneDrive/Coisas_Leonardo/gits/CurtosT2F/T2F_MATLAB/NovoArtigoPowerDelivery34bus/NovoModeloQualificacao/Teste_Novo_Sem_Terra_14/Processados_HDF5/MRT_SR__Sem_Falta_py.mat'
    ),
    (
        r'C:/Users/leosa/OneDrive/Coisas_Leonardo/gits/CurtosT2F/T2F_MATLAB/NovoArtigoPowerDelivery34bus/NovoModeloQualificacao/Teste_Novo_Sem_Terra_14/Processados_HDF5/Qualificacao__Sem_Falta_py.mat',
        r'C:/Users/leosa/OneDrive/Coisas_Leonardo/gits/CurtosT2F/T2F_MATLAB/NovoArtigoPowerDelivery34bus/NovoModeloQualificacao/Teste_Novo_Sem_Terra_14/Processados_HDF5/Qualificacao_SR__Sem_Falta_py.mat'
    ),
    (
        r'C:/Users/leosa/OneDrive/Coisas_Leonardo/gits/CurtosT2F/T2F_MATLAB/NovoArtigoPowerDelivery34bus/NovoModeloQualificacao/Teste_Novo_Sem_Terra_14/Processados_HDF5/MRT_sem_terra__Sem_Falta_py.mat',
        r'C:/Users/leosa/OneDrive/Coisas_Leonardo/gits/CurtosT2F/T2F_MATLAB/NovoArtigoPowerDelivery34bus/NovoModeloQualificacao/Teste_Novo_Sem_Terra_14/Processados_HDF5/MRT_SR_sem_terra__Sem_Falta_py.mat'
    ),
    (
        r'C:/Users/leosa/OneDrive/Coisas_Leonardo/gits/CurtosT2F/T2F_MATLAB/NovoArtigoPowerDelivery34bus/NovoModeloQualificacao/Teste_Novo_Sem_Terra_14/Processados_HDF5/Qualificacao_sem_terra__Sem_Falta_py.mat',
        r'C:/Users/leosa/OneDrive/Coisas_Leonardo/gits/CurtosT2F/T2F_MATLAB/NovoArtigoPowerDelivery34bus/NovoModeloQualificacao/Teste_Novo_Sem_Terra_14/Processados_HDF5/Qualificacao_SR_sem_terra__Sem_Falta_py.mat'
    ),
    (
        r'C:/Users/leosa/OneDrive/Coisas_Leonardo/gits/CurtosT2F/T2F_MATLAB/NovoArtigoPowerDelivery34bus/NovoModeloQualificacao/Teste_Novo_Sem_Terra_14/Processados_HDF5/MRT__Sem_Falta_py.mat',
        r'C:/Users/leosa/OneDrive/Coisas_Leonardo/gits/CurtosT2F/T2F_MATLAB/NovoArtigoPowerDelivery34bus/NovoModeloQualificacao/Teste_Novo_Sem_Terra_14/Processados_HDF5/Qualificacao__Sem_Falta_py.mat'
    ),
    (
        r'C:/Users/leosa/OneDrive/Coisas_Leonardo/gits/CurtosT2F/T2F_MATLAB/NovoArtigoPowerDelivery34bus/NovoModeloQualificacao/Teste_Novo_Sem_Terra_14/Processados_HDF5/MRT_SR__Sem_Falta_py.mat',
        r'C:/Users/leosa/OneDrive/Coisas_Leonardo/gits/CurtosT2F/T2F_MATLAB/NovoArtigoPowerDelivery34bus/NovoModeloQualificacao/Teste_Novo_Sem_Terra_14/Processados_HDF5/Qualificacao_SR__Sem_Falta_py.mat'
    ),
    (
        r'C:/Users/leosa/OneDrive/Coisas_Leonardo/gits/CurtosT2F/T2F_MATLAB/NovoArtigoPowerDelivery34bus/NovoModeloQualificacao/Teste_Novo_Sem_Terra_14/Processados_HDF5/MRT_sem_terra__Sem_Falta_py.mat',
        r'C:/Users/leosa/OneDrive/Coisas_Leonardo/gits/CurtosT2F/T2F_MATLAB/NovoArtigoPowerDelivery34bus/NovoModeloQualificacao/Teste_Novo_Sem_Terra_14/Processados_HDF5/Qualificacao_sem_terra__Sem_Falta_py.mat'
    ),
    (
        r'C:/Users/leosa/OneDrive/Coisas_Leonardo/gits/CurtosT2F/T2F_MATLAB/NovoArtigoPowerDelivery34bus/NovoModeloQualificacao/Teste_Novo_Sem_Terra_14/Processados_HDF5/MRT_SR_sem_terra__Sem_Falta_py.mat',
        r'C:/Users/leosa/OneDrive/Coisas_Leonardo/gits/CurtosT2F/T2F_MATLAB/NovoArtigoPowerDelivery34bus/NovoModeloQualificacao/Teste_Novo_Sem_Terra_14/Processados_HDF5/Qualificacao_SR_sem_terra__Sem_Falta_py.mat'
    )

]

tabela_indicadores = []

if lista_de_comparacoes:
    print(f"\nIniciando processamento avançado de {len(lista_de_comparacoes)} pares...\n")

    for i, (path1_str, path2_str) in enumerate(lista_de_comparacoes):
        arq1, arq2 = Path(path1_str), Path(path2_str)

        try:
            desc1 = descrever_arquivo(arq1.stem)
            desc2 = descrever_arquivo(arq2.stem)
            print(f"[{i+1}] Par: {arq1.stem}  vs  {arq2.stem}")
            print(f"      Caso A: {desc1}")
            print(f"      Caso B: {desc2}\n")

            # Legendas curtas para os gráficos
            l1, l2 = "Caso A", "Caso B"
            if "MRT" in arq1.stem:
                l1 = "MRT"
            if "Qualificacao" in arq1.stem:
                l1 = "T2F"
            if "_SR_" in arq1.stem:
                l1 += " SR"
            if "sem_terra" in arq1.stem:
                l1 += " s/terra"

            if "MRT" in arq2.stem:
                l2 = "MRT"
            if "Qualificacao" in arq2.stem:
                l2 = "T2F"
            if "_SR_" in arq2.stem:
                l2 += " SR"
            if "sem_terra" in arq2.stem:
                l2 += " s/terra"

            with h5py.File(arq1, 'r') as f1, h5py.File(arq2, 'r') as f2:
                t = np.array(f1['t']).flatten()

                vars_corr = ['I_800_raw', 'I_T2F_raw', 'I_818_raw', 'I_820_raw', 'I_822_raw']
                vars_tens = ['V_800_raw', 'V_T2F_raw', 'V_818_raw', 'V_820_raw', 'V_822_raw']

                # ---------------- CORRENTE ----------------
                for var in vars_corr:
                    d1 = ler_dados_padronizados(f1, var)
                    d2 = ler_dados_padronizados(f2, var)

                    if d1 is not None and d2 is not None:
                        i0_a, i1_a, i2_a = calcular_sequencias(d1, t)
                        i0_b, i1_b, i2_b = calcular_sequencias(d2, t)

                        # PASSAR O ID DO PAR (i+1)
                        plotar_comparacao_seq_zero(t, i0_a, i0_b, l1, l2, var, pasta_raiz_resultados, i+1)

                        imax_a = np.max(get_imax_envelope_vetor(t, d1))
                        imax_b = np.max(get_imax_envelope_vetor(t, d2))

                        # PASSAR O ID DO PAR (i+1)
                        plotar_tcc_coordenacao(imax_a, imax_b, l1, l2, var, pasta_raiz_resultados, i+1)

                        with np.errstate(divide='ignore', invalid='ignore'):
                            unb_a = np.max(np.nan_to_num(i2_a / i1_a)) * 100.0
                            unb_b = np.max(np.nan_to_num(i2_b / i1_b)) * 100.0

                        tabela_indicadores.append({
                            'Par': i+1,
                            'Barra': var,
                            'Tipo': 'Corrente',
                            'Caso A': desc1,
                            'Legenda_A': l1,
                            'Imax_A': imax_a,
                            'I0_Max_A': np.max(i0_a),
                            'Unbalance_A(%)': unb_a,
                            'Caso B': desc2,
                            'Legenda_B': l2,
                            'Imax_B': imax_b,
                            'I0_Max_B': np.max(i0_b),
                            'Unbalance_B(%)': unb_b
                        })

                # ---------------- TENSÃO ----------------
                for var in vars_tens:
                    d1 = ler_dados_padronizados(f1, var)
                    d2 = ler_dados_padronizados(f2, var)

                    if d1 is not None and d2 is not None:
                        dt = t[1] - t[0]
                        idx_pre = max(1, int(0.05 / dt))

                        rms_a_regime = np.mean(calcular_rms_movel(d1[:idx_pre, 0], t[:idx_pre]))
                        rms_b_regime = np.mean(calcular_rms_movel(d2[:idx_pre, 0], t[:idx_pre]))

                        # PASSAR O ID DO PAR (i+1)
                        plotar_perfil_tensao_regime(rms_a_regime, rms_b_regime, l1, l2, var, pasta_raiz_resultados, i+1)

                        v0_a, v1_a, v2_a = calcular_sequencias(d1, t)
                        v0_b, v1_b, v2_b = calcular_sequencias(d2, t)

                        with np.errstate(divide='ignore', invalid='ignore'):
                            vuf_a = np.max(np.nan_to_num(v2_a / v1_a)) * 100.0
                            vuf_b = np.max(np.nan_to_num(v2_b / v1_b)) * 100.0

                        tabela_indicadores.append({
                            'Par': i+1,
                            'Barra': var,
                            'Tipo': 'Tensao',
                            'Caso A': desc1,
                            'Legenda_A': l1,
                            'V_Regime_A': rms_a_regime,
                            'VUF_Max_A(%)': vuf_a,
                            'Caso B': desc2,
                            'Legenda_B': l2,
                            'V_Regime_B': rms_b_regime,
                            'VUF_Max_B(%)': vuf_b
                        })

        except Exception as e:
            print(f"ERRO no par {i+1}: {e}")
            import traceback
            traceback.print_exc()

    if tabela_indicadores:
        df = pd.DataFrame(tabela_indicadores)
        caminho_csv = pasta_raiz_resultados / "Tabela_Indicadores_Tese.csv"
        df.to_csv(caminho_csv, index=False, sep=';', decimal=',')
        print(f"\n--> Tabela de indicadores salva em: {caminho_csv}")
        print("--> Dica: abra o CSV no Excel usando 'Dados > Texto para Colunas' se necessário.")

    print("\n--- Processamento Completo Finalizado ---")


Iniciando processamento avançado de 8 pares...

[1] Par: MRT__Sem_Falta_py  vs  MRT_SR__Sem_Falta_py
      Caso A: MRT (Monofásico c/ Retorno por Terra/Neutro) | Com regulador | Com aterramento da subestação | Caso em regime (sem falta)
      Caso B: MRT (Monofásico c/ Retorno por Terra/Neutro) | Sem regulador | Com aterramento da subestação | Caso em regime (sem falta)

[2] Par: Qualificacao__Sem_Falta_py  vs  Qualificacao_SR__Sem_Falta_py
      Caso A: T2F (Trifásico a Dois Fios) | Com regulador | Com aterramento da subestação | Caso em regime (sem falta)
      Caso B: T2F (Trifásico a Dois Fios) | Sem regulador | Com aterramento da subestação | Caso em regime (sem falta)

[3] Par: MRT_sem_terra__Sem_Falta_py  vs  MRT_SR_sem_terra__Sem_Falta_py
      Caso A: MRT (Monofásico c/ Retorno por Terra/Neutro) | Com regulador | Sem aterramento da subestação | Caso em regime (sem falta)
      Caso B: MRT (Monofásico c/ Retorno por Terra/Neutro) | Sem regulador | Sem aterramento da subestação

In [16]:
# --- 1. CONFIGURAÇÕES ---
FREQ_SISTEMA = 60

# Configurações Matplotlib
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman"],
    "font.size": 12,
    "figure.figsize": cm_to_inch(16, 10),
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
    "savefig.transparent": False,
    "text.color": "black",
    "axes.labelcolor": "black",
    "xtick.color": "black",
    "ytick.color": "black",
    "axes.edgecolor": "black"
})

# -------------------------------------------------------------------
# 4. LOOP PRINCIPAL
# -------------------------------------------------------------------

lista_de_comparacoes = [
    (
        r'C:/Users/leosa/OneDrive/Coisas_Leonardo/gits/CurtosT2F/T2F_MATLAB/NovoArtigoPowerDelivery34bus/NovoModeloQualificacao/Teste_Novo_Sem_Terra_14/Processados_HDF5/MRT__Sem_Falta_py.mat',
        r'C:/Users/leosa/OneDrive/Coisas_Leonardo/gits/CurtosT2F/T2F_MATLAB/NovoArtigoPowerDelivery34bus/NovoModeloQualificacao/Teste_Novo_Sem_Terra_14/Processados_HDF5/MRT_SR__Sem_Falta_py.mat'
    ),
    (
        r'C:/Users/leosa/OneDrive/Coisas_Leonardo/gits/CurtosT2F/T2F_MATLAB/NovoArtigoPowerDelivery34bus/NovoModeloQualificacao/Teste_Novo_Sem_Terra_14/Processados_HDF5/Qualificacao__Sem_Falta_py.mat',
        r'C:/Users/leosa/OneDrive/Coisas_Leonardo/gits/CurtosT2F/T2F_MATLAB/NovoArtigoPowerDelivery34bus/NovoModeloQualificacao/Teste_Novo_Sem_Terra_14/Processados_HDF5/Qualificacao_SR__Sem_Falta_py.mat'
    ),
    (
        r'C:/Users/leosa/OneDrive/Coisas_Leonardo/gits/CurtosT2F/T2F_MATLAB/NovoArtigoPowerDelivery34bus/NovoModeloQualificacao/Teste_Novo_Sem_Terra_14/Processados_HDF5/MRT_sem_terra__Sem_Falta_py.mat',
        r'C:/Users/leosa/OneDrive/Coisas_Leonardo/gits/CurtosT2F/T2F_MATLAB/NovoArtigoPowerDelivery34bus/NovoModeloQualificacao/Teste_Novo_Sem_Terra_14/Processados_HDF5/MRT_SR_sem_terra__Sem_Falta_py.mat'
    ),
    (
        r'C:/Users/leosa/OneDrive/Coisas_Leonardo/gits/CurtosT2F/T2F_MATLAB/NovoArtigoPowerDelivery34bus/NovoModeloQualificacao/Teste_Novo_Sem_Terra_14/Processados_HDF5/Qualificacao_sem_terra__Sem_Falta_py.mat',
        r'C:/Users/leosa/OneDrive/Coisas_Leonardo/gits/CurtosT2F/T2F_MATLAB/NovoArtigoPowerDelivery34bus/NovoModeloQualificacao/Teste_Novo_Sem_Terra_14/Processados_HDF5/Qualificacao_SR_sem_terra__Sem_Falta_py.mat'
    ),
    (
        r'C:/Users/leosa/OneDrive/Coisas_Leonardo/gits/CurtosT2F/T2F_MATLAB/NovoArtigoPowerDelivery34bus/NovoModeloQualificacao/Teste_Novo_Sem_Terra_14/Processados_HDF5/MRT__Sem_Falta_py.mat',
        r'C:/Users/leosa/OneDrive/Coisas_Leonardo/gits/CurtosT2F/T2F_MATLAB/NovoArtigoPowerDelivery34bus/NovoModeloQualificacao/Teste_Novo_Sem_Terra_14/Processados_HDF5/Qualificacao__Sem_Falta_py.mat'
    ),
    (
        r'C:/Users/leosa/OneDrive/Coisas_Leonardo/gits/CurtosT2F/T2F_MATLAB/NovoArtigoPowerDelivery34bus/NovoModeloQualificacao/Teste_Novo_Sem_Terra_14/Processados_HDF5/MRT_SR__Sem_Falta_py.mat',
        r'C:/Users/leosa/OneDrive/Coisas_Leonardo/gits/CurtosT2F/T2F_MATLAB/NovoArtigoPowerDelivery34bus/NovoModeloQualificacao/Teste_Novo_Sem_Terra_14/Processados_HDF5/Qualificacao_SR__Sem_Falta_py.mat'
    ),
    (
        r'C:/Users/leosa/OneDrive/Coisas_Leonardo/gits/CurtosT2F/T2F_MATLAB/NovoArtigoPowerDelivery34bus/NovoModeloQualificacao/Teste_Novo_Sem_Terra_14/Processados_HDF5/MRT_sem_terra__Sem_Falta_py.mat',
        r'C:/Users/leosa/OneDrive/Coisas_Leonardo/gits/CurtosT2F/T2F_MATLAB/NovoArtigoPowerDelivery34bus/NovoModeloQualificacao/Teste_Novo_Sem_Terra_14/Processados_HDF5/Qualificacao_sem_terra__Sem_Falta_py.mat'
    ),
    (
        r'C:/Users/leosa/OneDrive/Coisas_Leonardo/gits/CurtosT2F/T2F_MATLAB/NovoArtigoPowerDelivery34bus/NovoModeloQualificacao/Teste_Novo_Sem_Terra_14/Processados_HDF5/MRT_SR_sem_terra__Sem_Falta_py.mat',
        r'C:/Users/leosa/OneDrive/Coisas_Leonardo/gits/CurtosT2F/T2F_MATLAB/NovoArtigoPowerDelivery34bus/NovoModeloQualificacao/Teste_Novo_Sem_Terra_14/Processados_HDF5/Qualificacao_SR_sem_terra__Sem_Falta_py.mat'
    )

]

# -------------------------------------------------------------------
# 1.A – FUNÇÃO PARA "TRADUZIR" O NOME DO ARQUIVO
# -------------------------------------------------------------------

def descrever_arquivo(stem: str) -> str:
    partes = []
    if "MRT" in stem:
        partes.append("MRT (Monofásico c/ Retorno por Terra/Neutro)")
    elif "Qualificacao" in stem:
        partes.append("T2F (Trifásico a Dois Fios)")
    else:
        partes.append("Topologia não identificada")

    if "_SR_" in stem:
        partes.append("Sem regulador")
    else:
        partes.append("Com regulador")

    if "sem_terra" in stem:
        partes.append("Sem aterramento da subestação")
    else:
        partes.append("Com aterramento da subestação")

    if "Sem_Falta" in stem:
        partes.append("Caso em regime (sem falta)")
    else:
        partes.append("Caso com falta (não explicitado)")

    return " | ".join(partes)

# -------------------------------------------------------------------
# 2. FUNÇÕES MATEMÁTICAS (RMS, FASORES, COMPONENTES, TCC, FFT)
# -------------------------------------------------------------------

def calcular_rms_movel(sinal, tempo, freq_rede=60):
    if len(tempo) < 2:
        return np.zeros_like(sinal)
    dt = tempo[1] - tempo[0]
    if dt <= 0:
        return np.zeros_like(sinal)
    fs = 1.0 / dt
    janela = int(fs / freq_rede)
    if janela < 1:
        janela = 1
    sinal_quadrado = sinal ** 2
    janela_media = np.ones(janela) / janela
    return np.sqrt(np.convolve(sinal_quadrado, janela_media, mode='same'))

def extrair_fasor_dinamico(sinal, tempo, freq=60):
    if len(tempo) < 2:
        return np.zeros_like(sinal, dtype=complex)
    dt = tempo[1] - tempo[0]
    if dt <= 0:
        return np.zeros_like(sinal, dtype=complex)

    samples = int((1.0/freq) / dt)
    if samples < 1:
        samples = 1

    t_win = np.arange(samples) * dt
    k_cos = np.cos(2*np.pi*freq*t_win) * (2.0/samples)
    k_sin = np.sin(2*np.pi*freq*t_win) * (2.0/samples)

    real = np.convolve(sinal, k_cos, mode='same')
    imag = np.convolve(sinal, k_sin, mode='same')
    return real - 1j*imag

def calcular_sequencias(dados_3fases, tempo, freq=60):
    Va = extrair_fasor_dinamico(dados_3fases[:, 0], tempo, freq)
    Vb = extrair_fasor_dinamico(dados_3fases[:, 1], tempo, freq)
    Vc = extrair_fasor_dinamico(dados_3fases[:, 2], tempo, freq)

    a = np.exp(1j * 2 * np.pi / 3)

    V0 = (Va + Vb + Vc) / 3.0
    V1 = (Va + a*Vb + a**2 * Vc) / 3.0
    V2 = (Va + a**2 * Vb + a * Vc) / 3.0

    return np.abs(V0), np.abs(V1), np.abs(V2)

def calcular_fft_harmonicas(sinal, tempo, freq_fund=60):
    """
    Calcula FFT e retorna magnitude das harmônicas de interesse.
    Retorna: dict com 'fundamental', 'h3', 'h5', 'h7', 'thd'
    """
    if len(sinal) < 2 or len(tempo) < 2:
        return {'fundamental': 0, 'h3': 0, 'h5': 0, 'h7': 0, 'thd': 0}

    dt = tempo[1] - tempo[0]
    n = len(sinal)

    # Remove componente DC
    sinal_ac = sinal - np.mean(sinal)

    # **MELHORIA 1**: Verifica se sinal é significativo
    rms_signal = np.sqrt(np.mean(sinal_ac**2))
    if rms_signal < 1e-6:  # Sinal muito baixo (ruído)
        return {'fundamental': 0, 'h3': 0, 'h5': 0, 'h7': 0, 'thd': 0, 'nota': 'Sinal desprezível'}

    # Aplica janela de Hanning
    window = np.hanning(n)
    sinal_windowed = sinal_ac * window

    # FFT
    fft_vals = np.fft.fft(sinal_windowed)
    freqs = np.fft.fftfreq(n, dt)

    # Magnitude (apenas metade positiva)
    mags = 2.0 * np.abs(fft_vals[:n//2]) / n
    freqs_pos = freqs[:n//2]

    # **MELHORIA 2**: Busca com tolerância de frequência
    def find_harmonic(h_order, tolerance_hz=5):
        target_freq = h_order * freq_fund
        # Encontra pico na faixa [target - tol, target + tol]
        mask = (freqs_pos >= target_freq - tolerance_hz) & (freqs_pos <= target_freq + tolerance_hz)
        if np.any(mask):
            return np.max(mags[mask])
        return 0.0

    h1 = find_harmonic(1)  # Fundamental
    h3 = find_harmonic(3)
    h5 = find_harmonic(5)
    h7 = find_harmonic(7)
    h9 = find_harmonic(9)
    h11 = find_harmonic(11)
    h13 = find_harmonic(13)

    # **MELHORIA 3**: THD só se fundamental for significativo
    thd = 0.0
    thd_completo = 0.0

    if h1 > 1e-3:  # Fundamental deve ser > 1 mA
        # THD simplificado (até 7ª)
        thd = 100.0 * np.sqrt(h3**2 + h5**2 + h7**2) / h1

        # THD mais completo (até 13ª)
        thd_completo = 100.0 * np.sqrt(h3**2 + h5**2 + h7**2 + h9**2 + h11**2 + h13**2) / h1
    else:
        thd = 0.0  # Marca como não aplicável
        thd_completo = 0.0

    return {
        'fundamental': h1,
        'h3': h3,
        'h5': h5,
        'h7': h7,
        'h9': h9,
        'h11': h11,
        'h13': h13,
        'thd': thd,
        'thd_completo': thd_completo
    }

def calcular_tempo_tcc_iec(corrente, i_pickup=10, time_dial=0.5, curve_type='VI'):
    curves = {
        'SI': (0.14, 0.02),
        'VI': (13.5, 1.0),
        'EI': (80.0, 2.0)
    }
    k, alpha = curves.get(curve_type, (13.5, 1.0))

    if corrente <= 0 or corrente < i_pickup:
        return None

    M = corrente / i_pickup
    if M <= 1.0:
        return None

    t = time_dial * (k / (M**alpha - 1.0))
    return t

def ler_dados_padronizados(f_obj, var_nome):
    if var_nome not in f_obj:
        return None
    dados_raw = np.array(f_obj[var_nome])

    if dados_raw.ndim == 2:
        if dados_raw.shape[0] < dados_raw.shape[1]:
            dados_raw = dados_raw.T
    elif dados_raw.ndim == 1:
        dados_raw = dados_raw.reshape(-1, 1)

    linhas, colunas = dados_raw.shape
    dados_pad = np.zeros((linhas, 3))
    for c in range(min(colunas, 3)):
        dados_pad[:, c] = dados_raw[:, c]
    return dados_pad

def get_imax_envelope_vetor(tempo, dados):
    rms_a = calcular_rms_movel(dados[:, 0], tempo)
    rms_b = calcular_rms_movel(dados[:, 1], tempo)
    rms_c = calcular_rms_movel(dados[:, 2], tempo)
    return np.maximum.reduce([rms_a, rms_b, rms_c])

# -------------------------------------------------------------------
# 3. FUNÇÕES DE PLOTAGEM
# -------------------------------------------------------------------

def plotar_comparacao_seq_zero(t, i0_a, i0_b, lab_a, lab_b, nome_barra, pasta, par_id):
    plt.figure()
    plt.plot(t, i0_a, color='black', label=f'{lab_a} (I0)', linewidth=1.5)
    plt.plot(t, i0_b, color='red', label=f'{lab_b} (I0)', linewidth=1.5, linestyle='--')
    plt.xlabel('Tempo (s)')
    plt.ylabel('Corrente de Sequência Zero (A)')
    plt.title(f'Comparação I0 - {nome_barra.replace("_raw","")}')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.tight_layout()
    plt.savefig(pasta / f"Par{par_id:02d}_SeqZero_{nome_barra}.svg", format='svg', facecolor='white')
    plt.close()

def plotar_espectro_harmonico(harm_a, harm_b, lab_a, lab_b, nome_barra, pasta, par_id):
    """
    Plota espectro de harmônicas (Fund, 3ª, 5ª, 7ª) para os dois casos.
    """
    harmonics = ['Fund\n(60Hz)', '3ª\n(180Hz)', '5ª\n(300Hz)', '7ª\n(420Hz)']
    vals_a = [harm_a['fundamental'], harm_a['h3'], harm_a['h5'], harm_a['h7']]
    vals_b = [harm_b['fundamental'], harm_b['h3'], harm_b['h5'], harm_b['h7']]

    x = np.arange(len(harmonics))
    width = 0.35

    fig, ax = plt.subplots()
    bars1 = ax.bar(x - width/2, vals_a, width, label=lab_a, color='blue', edgecolor='black')
    bars2 = ax.bar(x + width/2, vals_b, width, label=lab_b, color='red', edgecolor='black', alpha=0.7)

    ax.set_ylabel('Magnitude (A)')
    ax.set_title(f'Espectro Harmônico I0 - {nome_barra.replace("_raw","")}')
    ax.set_xticks(x)
    ax.set_xticklabels(harmonics)
    ax.legend()
    ax.grid(axis='y', linestyle='--', alpha=0.5)

    # Adiciona THD
    ax.text(0.02, 0.98, f'THD {lab_a}: {harm_a["thd"]:.2f}%\nTHD {lab_b}: {harm_b["thd"]:.2f}%',
            transform=ax.transAxes, fontsize=10, verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

    plt.tight_layout()
    plt.savefig(pasta / f"Par{par_id:02d}_FFT_{nome_barra}.svg", format='svg', facecolor='white')
    plt.close()

def plotar_tcc_coordenacao(imax_a, imax_b, lab_a, lab_b, nome_barra, pasta, par_id):
    I_PICKUP = 50.0
    TIME_DIAL = 0.5
    TIPO_CURVA = 'VI'

    imax_global = max(imax_a, imax_b)
    if imax_global <= I_PICKUP * 1.1:
        return

    correntes_x = np.logspace(
        np.log10(I_PICKUP*1.1),
        np.log10(imax_global*1.5),
        200
    )
    tempos_y = []
    for i in correntes_x:
        t_val = calcular_tempo_tcc_iec(i, I_PICKUP, TIME_DIAL, TIPO_CURVA)
        if t_val is None or t_val <= 0:
            t_val = np.nan
        tempos_y.append(t_val)
    tempos_y = np.array(tempos_y)

    plt.figure()
    plt.loglog(correntes_x, tempos_y, 'k-', label=f'IEC {TIPO_CURVA} (TD={TIME_DIAL})')

    t_a = calcular_tempo_tcc_iec(imax_a, I_PICKUP, TIME_DIAL, TIPO_CURVA)
    if t_a is not None and t_a > 0:
        plt.loglog(imax_a, t_a, 'bo', markersize=8,
                   label=f'{lab_a}: {imax_a:.0f} A / {t_a:.2f} s')

    t_b = calcular_tempo_tcc_iec(imax_b, I_PICKUP, TIME_DIAL, TIPO_CURVA)
    if t_b is not None and t_b > 0:
        plt.loglog(imax_b, t_b, 'rx', markersize=8, markeredgewidth=2,
                   label=f'{lab_b}: {imax_b:.0f} A / {t_b:.2f} s')

    plt.xlabel('Corrente (A)')
    plt.ylabel('Tempo de Atuação (s)')
    plt.title(f'Coordenação TCC - {nome_barra.replace("_raw","")}')
    plt.grid(True, which="both", ls="--", alpha=0.4)
    plt.legend()
    plt.tight_layout()
    plt.savefig(pasta / f"Par{par_id:02d}_TCC_{nome_barra}.svg", format='svg', facecolor='white')
    plt.close()

def plotar_perfil_tensao_regime(val_a, val_b, lab_a, lab_b, nome_barra, pasta, par_id):
    plt.figure()
    plt.bar([lab_a, lab_b], [val_a, val_b],
            color=['blue', 'red'], edgecolor='black', width=0.5)
    plt.ylabel('Tensão Pré-Falta (V)')
    plt.title(f'Perfil de Tensão - {nome_barra.replace("_raw","")}')
    plt.ylim(0, max(val_a, val_b)*1.2 if max(val_a, val_b) > 0 else 1)

    for i, v in enumerate([val_a, val_b]):
        plt.text(i, v*1.01, f'{v:.1f} V', ha='center',
                 color='black', fontweight='bold')

    plt.tight_layout()
    plt.savefig(pasta / f"Par{par_id:02d}_PerfilTensao_{nome_barra}.svg",
                format='svg', facecolor='white')
    plt.close()

def plotar_comparacao_seq_zero(t, i0_a, i0_b, lab_a, lab_b, nome_barra, pasta, par_id):
    plt.figure()
    plt.plot(t, i0_a, color='black', label=f'{lab_a} (I0)', linewidth=1.5)
    plt.plot(t, i0_b, color='red', label=f'{lab_b} (I0)', linewidth=1.5, linestyle='--')
    plt.xlabel('Tempo (s)')
    plt.ylabel('Corrente de Sequência Zero (A)')
    # INCLUIR PAR NO TÍTULO
    plt.title(f'Par {par_id} - Comparação I0 - {nome_barra.replace("_raw","")}')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.tight_layout()
    plt.savefig(pasta / f"Par{par_id:02d}_SeqZero_{nome_barra}.svg", format='svg', facecolor='white')
    plt.close()

def plotar_espectro_harmonico(harm_a, harm_b, lab_a, lab_b, nome_barra, pasta, par_id):
    """
    Plota espectro de harmônicas (Fund, 3ª, 5ª, 7ª) para os dois casos.
    """
    harmonics = ['Fund\n(60Hz)', '3ª\n(180Hz)', '5ª\n(300Hz)', '7ª\n(420Hz)']
    vals_a = [harm_a['fundamental'], harm_a['h3'], harm_a['h5'], harm_a['h7']]
    vals_b = [harm_b['fundamental'], harm_b['h3'], harm_b['h5'], harm_b['h7']]

    x = np.arange(len(harmonics))
    width = 0.35

    fig, ax = plt.subplots()
    bars1 = ax.bar(x - width/2, vals_a, width, label=lab_a, color='blue', edgecolor='black')
    bars2 = ax.bar(x + width/2, vals_b, width, label=lab_b, color='red', edgecolor='black', alpha=0.7)

    ax.set_ylabel('Magnitude (A)')
    # INCLUIR PAR NO TÍTULO
    ax.set_title(f'Par {par_id} - Espectro Harmônico I0 - {nome_barra.replace("_raw","")}')
    ax.set_xticks(x)
    ax.set_xticklabels(harmonics)
    ax.legend()
    ax.grid(axis='y', linestyle='--', alpha=0.5)

    # Adiciona THD
    ax.text(0.02, 0.98, f'THD {lab_a}: {harm_a["thd"]:.2f}%\nTHD {lab_b}: {harm_b["thd"]:.2f}%',
            transform=ax.transAxes, fontsize=10, verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

    plt.tight_layout()
    plt.savefig(pasta / f"Par{par_id:02d}_FFT_{nome_barra}.svg", format='svg', facecolor='white')
    plt.close()

def plotar_tcc_coordenacao(imax_a, imax_b, lab_a, lab_b, nome_barra, pasta, par_id):
    I_PICKUP = 50.0
    TIME_DIAL = 0.5
    TIPO_CURVA = 'VI'

    imax_global = max(imax_a, imax_b)
    if imax_global <= I_PICKUP * 1.1:
        return

    correntes_x = np.logspace(
        np.log10(I_PICKUP*1.1),
        np.log10(imax_global*1.5),
        200
    )
    tempos_y = []
    for i in correntes_x:
        t_val = calcular_tempo_tcc_iec(i, I_PICKUP, TIME_DIAL, TIPO_CURVA)
        if t_val is None or t_val <= 0:
            t_val = np.nan
        tempos_y.append(t_val)
    tempos_y = np.array(tempos_y)

    plt.figure()
    plt.loglog(correntes_x, tempos_y, 'k-', label=f'IEC {TIPO_CURVA} (TD={TIME_DIAL})')

    t_a = calcular_tempo_tcc_iec(imax_a, I_PICKUP, TIME_DIAL, TIPO_CURVA)
    if t_a is not None and t_a > 0:
        plt.loglog(imax_a, t_a, 'bo', markersize=8,
                   label=f'{lab_a}: {imax_a:.0f} A / {t_a:.2f} s')

    t_b = calcular_tempo_tcc_iec(imax_b, I_PICKUP, TIME_DIAL, TIPO_CURVA)
    if t_b is not None and t_b > 0:
        plt.loglog(imax_b, t_b, 'rx', markersize=8, markeredgewidth=2,
                   label=f'{lab_b}: {imax_b:.0f} A / {t_b:.2f} s')

    plt.xlabel('Corrente (A)')
    plt.ylabel('Tempo de Atuação (s)')
    # INCLUIR PAR NO TÍTULO
    plt.title(f'Par {par_id} - Coordenação TCC - {nome_barra.replace("_raw","")}')
    plt.grid(True, which="both", ls="--", alpha=0.4)
    plt.legend()
    plt.tight_layout()
    plt.savefig(pasta / f"Par{par_id:02d}_TCC_{nome_barra}.svg", format='svg', facecolor='white')
    plt.close()

def plotar_perfil_tensao_regime(val_a, val_b, lab_a, lab_b, nome_barra, pasta, par_id):
    plt.figure()
    plt.bar([lab_a, lab_b], [val_a, val_b],
            color=['blue', 'red'], edgecolor='black', width=0.5)
    plt.ylabel('Tensão Pré-Falta (V)')
    # INCLUIR PAR NO TÍTULO
    plt.title(f'Par {par_id} - Perfil de Tensão - {nome_barra.replace("_raw","")}')
    plt.ylim(0, max(val_a, val_b)*1.2 if max(val_a, val_b) > 0 else 1)

    for i, v in enumerate([val_a, val_b]):
        plt.text(i, v*1.01, f'{v:.1f} V', ha='center',
                 color='black', fontweight='bold')

    plt.tight_layout()
    plt.savefig(pasta / f"Par{par_id:02d}_PerfilTensao_{nome_barra}.svg",
                format='svg', facecolor='white')
    plt.close()

tabela_indicadores = []

if lista_de_comparacoes:
    print(f"\nIniciando processamento avançado de {len(lista_de_comparacoes)} pares...\n")

    for i, (path1_str, path2_str) in enumerate(lista_de_comparacoes):
        arq1, arq2 = Path(path1_str), Path(path2_str)

        try:
            desc1 = descrever_arquivo(arq1.stem)
            desc2 = descrever_arquivo(arq2.stem)
            print(f"[{i+1}] Par: {arq1.stem}  vs  {arq2.stem}")
            print(f"      Caso A: {desc1}")
            print(f"      Caso B: {desc2}\n")

            # Legendas curtas para os gráficos
            l1, l2 = "Caso A", "Caso B"
            if "MRT" in arq1.stem:
                l1 = "MRT"
            if "Qualificacao" in arq1.stem:
                l1 = "T2F"
            if "_SR_" in arq1.stem:
                l1 += " SR"
            if "sem_terra" in arq1.stem:
                l1 += " s/terra"

            if "MRT" in arq2.stem:
                l2 = "MRT"
            if "Qualificacao" in arq2.stem:
                l2 = "T2F"
            if "_SR_" in arq2.stem:
                l2 += " SR"
            if "sem_terra" in arq2.stem:
                l2 += " s/terra"

            with h5py.File(arq1, 'r') as f1, h5py.File(arq2, 'r') as f2:
                t = np.array(f1['t']).flatten()

                vars_corr = ['I_800_raw', 'I_T2F_raw', 'I_818_raw', 'I_820_raw', 'I_822_raw']
                vars_tens = ['V_800_raw', 'V_T2F_raw', 'V_818_raw', 'V_820_raw', 'V_822_raw']

                # ---------------- CORRENTE ----------------
                for var in vars_corr:
                    d1 = ler_dados_padronizados(f1, var)
                    d2 = ler_dados_padronizados(f2, var)

                    if d1 is not None and d2 is not None:
                        # **DIAGNÓSTICO**: Verifica se dados são válidos
                        max_d1 = np.max(np.abs(d1))
                        max_d2 = np.max(np.abs(d2))
                        print(f"    {var}: max |d1|={max_d1:.2e} A, max |d2|={max_d2:.2e} A")

                        i0_a, i1_a, i2_a = calcular_sequencias(d1, t)
                        i0_b, i1_b, i2_b = calcular_sequencias(d2, t)

                        # Verifica se I0 tem conteúdo
                        max_i0_a = np.max(i0_a)
                        max_i0_b = np.max(i0_b)
                        print(f"    {var}: max I0_a={max_i0_a:.2e} A, max I0_b={max_i0_b:.2e} A")

                        # Plot I0 temporal
                        plotar_comparacao_seq_zero(t, i0_a, i0_b, l1, l2, var, pasta_raiz_resultados, i+1)

                        # **FFT corrigido**
                        dt = t[1] - t[0] if len(t) > 1 else 0.0001

                        # Usa janela após transitório (0.1s a 0.4s para ter dados suficientes)
                        idx_inicio = int(0.1 / dt)
                        idx_fim = int(0.4 / dt)
                        idx_inicio = max(0, min(idx_inicio, len(t)-2))
                        idx_fim = min(idx_fim, len(t))

                        if idx_fim > idx_inicio + 10:  # Precisa de ao menos 10 pontos
                            # Pega janela de regime
                            i0_a_regime = i0_a[idx_inicio:idx_fim]
                            i0_b_regime = i0_b[idx_inicio:idx_fim]
                            t_regime = t[idx_inicio:idx_fim]

                            print(f"    FFT em [{t_regime[0]:.3f}s - {t_regime[-1]:.3f}s], {len(i0_a_regime)} amostras")

                            harm_a = calcular_fft_harmonicas(i0_a_regime, t_regime, FREQ_SISTEMA)
                            harm_b = calcular_fft_harmonicas(i0_b_regime, t_regime, FREQ_SISTEMA)

                            print(f"    THD_A = {harm_a['thd']:.2f}%, THD_B = {harm_b['thd']:.2f}%")

                            # Plot espectro harmônico
                            plotar_espectro_harmonico(harm_a, harm_b, l1, l2, var, pasta_raiz_resultados, i+1)
                        else:
                            harm_a = {'fundamental': 0, 'h3': 0, 'h5': 0, 'h7': 0, 'h7': 0, 'thd': 0}
                            harm_b = {'fundamental': 0, 'h3': 0, 'h5': 0, 'h7': 0, 'h9': 0, 'thd': 0}
                            print(f"    AVISO: Janela de regime insuficiente para FFT")

                        # Envelope máximo
                        imax_a = np.max(get_imax_envelope_vetor(t, d1))
                        imax_b = np.max(get_imax_envelope_vetor(t, d2))

                        # TCC
                        plotar_tcc_coordenacao(imax_a, imax_b, l1, l2, var, pasta_raiz_resultados, i+1)

                        # Indicadores agregados
                        with np.errstate(divide='ignore', invalid='ignore'):
                            unb_a = np.max(np.nan_to_num(i2_a / i1_a)) * 100.0
                            unb_b = np.max(np.nan_to_num(i2_b / i1_b)) * 100.0

                        tabela_indicadores.append({
                            'Par': i+1,
                            'Barra': var,
                            'Tipo': 'Corrente',
                            'Caso A': desc1,
                            'Legenda_A': l1,
                            'Imax_A': imax_a,
                            'I0_Max_A': np.max(i0_a),
                            'Unbalance_A(%)': unb_a,
                            'H3_A': harm_a['h3'],
                            'H5_A': harm_a['h5'],
                            'H7_A': harm_a['h7'],
                            'THD_A(%)': harm_a['thd'],
                            'Caso B': desc2,
                            'Legenda_B': l2,
                            'Imax_B': imax_b,
                            'I0_Max_B': np.max(i0_b),
                            'Unbalance_B(%)': unb_b,
                            'H3_B': harm_b['h3'],
                            'H5_B': harm_b['h5'],
                            'H7_B': harm_b['h7'],
                            'THD_B(%)': harm_b['thd']
                        })

                # ---------------- TENSÃO ----------------
                for var in vars_tens:
                    d1 = ler_dados_padronizados(f1, var)
                    d2 = ler_dados_padronizados(f2, var)

                    if d1 is not None and d2 is not None:
                        dt = t[1] - t[0]
                        idx_pre = max(1, int(0.05 / dt))

                        rms_a_regime = np.mean(calcular_rms_movel(d1[:idx_pre, 0], t[:idx_pre]))
                        rms_b_regime = np.mean(calcular_rms_movel(d2[:idx_pre, 0], t[:idx_pre]))

                        # Perfil de tensão
                        plotar_perfil_tensao_regime(rms_a_regime, rms_b_regime, l1, l2, var, pasta_raiz_resultados, i+1)

                        # Componentes e VUF
                        v0_a, v1_a, v2_a = calcular_sequencias(d1, t)
                        v0_b, v1_b, v2_b = calcular_sequencias(d2, t)

                        with np.errstate(divide='ignore', invalid='ignore'):
                            vuf_a = np.max(np.nan_to_num(v2_a / v1_a)) * 100.0
                            vuf_b = np.max(np.nan_to_num(v2_b / v1_b)) * 100.0

                        tabela_indicadores.append({
                            'Par': i+1,
                            'Barra': var,
                            'Tipo': 'Tensao',
                            'Caso A': desc1,
                            'Legenda_A': l1,
                            'V_Regime_A': rms_a_regime,
                            'VUF_Max_A(%)': vuf_a,
                            'Caso B': desc2,
                            'Legenda_B': l2,
                            'V_Regime_B': rms_b_regime,
                            'VUF_Max_B(%)': vuf_b
                        })

        except Exception as e:
            print(f"ERRO no par {i+1}: {e}")
            import traceback
            traceback.print_exc()

    # ---------------- SALVA TABELA ----------------
    if tabela_indicadores:
        df = pd.DataFrame(tabela_indicadores)
        caminho_csv = pasta_raiz_resultados / "Tabela_Indicadores_Tese.csv"
        df.to_csv(caminho_csv, index=False, sep=';', decimal=',')
        print(f"\n--> Tabela de indicadores salva em: {caminho_csv}")
        print("--> Dica: abra o CSV no Excel usando 'Dados > Texto para Colunas' se necessário.")

    print("\n--- Processamento Completo Finalizado ---")


Iniciando processamento avançado de 8 pares...

[1] Par: MRT__Sem_Falta_py  vs  MRT_SR__Sem_Falta_py
      Caso A: MRT (Monofásico c/ Retorno por Terra/Neutro) | Com regulador | Com aterramento da subestação | Caso em regime (sem falta)
      Caso B: MRT (Monofásico c/ Retorno por Terra/Neutro) | Sem regulador | Com aterramento da subestação | Caso em regime (sem falta)

    I_800_raw: max |d1|=2.47e+01 A, max |d2|=2.28e+01 A
    I_800_raw: max I0_a=6.76e+00 A, max I0_b=6.27e+00 A
    FFT em [0.100s - 0.400s], 6000 amostras
    THD_A = 0.00%, THD_B = 0.00%
    I_T2F_raw: max |d1|=2.04e+01 A, max |d2|=2.06e+01 A
    I_T2F_raw: max I0_a=6.23e+00 A, max I0_b=6.23e+00 A
    FFT em [0.100s - 0.400s], 6000 amostras
    THD_A = 0.00%, THD_B = 0.00%
    I_818_raw: max |d1|=2.04e+01 A, max |d2|=2.05e+01 A
    I_818_raw: max I0_a=6.23e+00 A, max I0_b=6.23e+00 A
    FFT em [0.100s - 0.400s], 6000 amostras
    THD_A = 0.00%, THD_B = 0.00%
    I_820_raw: max |d1|=1.54e+01 A, max |d2|=1.55e+01 A
  

In [10]:
# ======================================================================
# SCRIPT DE PROCESSAMENTO – CAPÍTULO 5 (SELEÇÃO FLEXÍVEL DE BARRAS)
# ======================================================================

import h5py
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
import csv

# ----------------------------------------------------------------------
# 1. CONFIGURAÇÕES GERAIS
# ----------------------------------------------------------------------

FREQUENCIA_REDE = 60  # Hz

def cm_to_inch(value):
    return value / 2.54

plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman"],
    "font.size": 12,
    "figure.figsize": (cm_to_inch(16), cm_to_inch(10)),
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
    "savefig.transparent": False,
    "text.color": "black",
    "axes.labelcolor": "black",
    "xtick.color": "black",
    "ytick.color": "black",
    "axes.edgecolor": "black"
})

pasta_raiz_resultados = Path("Resultados_Tese_Cap5")

# ----------------------------------------------------------------------
# 2. FUNÇÕES MATEMÁTICAS AUXILIARES
# ----------------------------------------------------------------------

def calcular_rms_movel(sinal, tempo, freq_rede=FREQUENCIA_REDE):
    if len(tempo) < 2:
        return np.zeros_like(sinal)
    dt = tempo[1] - tempo[0]
    if dt <= 0:
        return np.zeros_like(sinal)
    fs = 1.0 / dt
    janela = int(fs / freq_rede)
    if janela < 1:
        janela = 1
    sinal_quadrado = sinal**2
    janela_media = np.ones(janela) / janela
    return np.sqrt(np.convolve(sinal_quadrado, janela_media, mode="same"))

def calcular_fft_para_plot(sinal, tempo):
    dt = tempo[1] - tempo[0]
    n = len(sinal)
    fhat = np.fft.fft(sinal)
    freqs = np.fft.fftfreq(n, d=dt)
    mags = 2 * np.abs(fhat) / n
    mask = freqs >= 0
    return freqs[mask], mags[mask]

def calcular_v1_v3_global(sinal, tempo, freq_rede=FREQUENCIA_REDE):
    dt = tempo[1] - tempo[0]
    n = len(sinal)
    fhat = np.fft.fft(sinal)
    freqs = np.fft.fftfreq(n, d=dt)
    mags = 2 * np.abs(fhat) / n
    idx_60 = np.argmin(np.abs(freqs - freq_rede))
    idx_180 = np.argmin(np.abs(freqs - 3 * freq_rede))
    return float(mags[idx_60]), float(mags[idx_180])

def get_imax_envelope_vetor(tempo, dados_tres_fases):
    rms_a = calcular_rms_movel(dados_tres_fases[:, 0], tempo)
    rms_b = calcular_rms_movel(dados_tres_fases[:, 1], tempo)
    rms_c = calcular_rms_movel(dados_tres_fases[:, 2], tempo)
    return np.maximum.reduce([rms_a, rms_b, rms_c])

def extrair_fasor_dinamico(sinal, tempo, freq=FREQUENCIA_REDE):
    dt = tempo[1] - tempo[0]
    if dt <= 0:
        return np.zeros_like(sinal, dtype=complex)
    samples_per_cycle = int((1.0 / freq) / dt)
    if samples_per_cycle < 1:
        samples_per_cycle = 1
    t_window = np.arange(samples_per_cycle) * dt
    kernel_cos = np.cos(2 * np.pi * freq * t_window) * (2 / samples_per_cycle)
    kernel_sin = np.sin(2 * np.pi * freq * t_window) * (2 / samples_per_cycle)
    real_part = np.convolve(sinal, kernel_cos, mode="same")
    imag_part = np.convolve(sinal, kernel_sin, mode="same")
    return real_part - 1j * imag_part

def calcular_componentes_simetricas_tempo(dados_3fases, tempo, freq=FREQUENCIA_REDE):
    Va = extrair_fasor_dinamico(dados_3fases[:, 0], tempo, freq)
    Vb = extrair_fasor_dinamico(dados_3fases[:, 1], tempo, freq)
    Vc = extrair_fasor_dinamico(dados_3fases[:, 2], tempo, freq)
    a = np.exp(1j * 2 * np.pi / 3)
    a2 = a**2
    V0 = (Va + Vb + Vc) / 3
    V1 = (Va + a * Vb + a2 * Vc) / 3
    V2 = (Va + a2 * Vb + a * Vc) / 3
    return np.abs(V0), np.abs(V1), np.abs(V2)

# ---- curvas TCC IEC/IEEE -----------------------------------------------------

def t_iec(M, TMS, k, c, alpha):
    return TMS * k / (M**alpha - 1.0) + c

def t_ieee(M, TD, k, c, alpha):
    return TD * k / (M**alpha - 1.0) + c

def tempo_atuacao_religador(I_falta, I_pickup, T_scalar, curva_tipo="IEC_C1"):
    M = I_falta / I_pickup
    if M <= 1.0:
        return np.inf
    if curva_tipo == "IEC_C1":
        k, c, alpha = 0.14, 0.0, 0.02
        return t_iec(M, T_scalar, k, c, alpha)
    elif curva_tipo == "IEC_C2":
        k, c, alpha = 13.5, 0.0, 1.0
        return t_iec(M, T_scalar, k, c, alpha)
    elif curva_tipo == "IEC_C3":
        k, c, alpha = 80.0, 0.0, 2.0
        return t_iec(M, T_scalar, k, c, alpha)
    elif curva_tipo == "IEEE_U1":
        k, c, alpha = 0.0515, 0.114, 0.02
        return t_ieee(M, T_scalar, k, c, alpha)
    elif curva_tipo == "IEEE_U2":
        k, c, alpha = 19.61, 0.491, 2.0
        return t_ieee(M, T_scalar, k, c, alpha)
    elif curva_tipo == "IEEE_U3":
        k, c, alpha = 28.2, 0.1217, 2.0
        return t_ieee(M, T_scalar, k, c, alpha)
    else:
        raise ValueError(f"Tipo de curva não reconhecido: {curva_tipo}")

def plotar_curvas_tcc_religadores(caminho_salvar):
    M = np.logspace(0, 2, 400)
    TMS = 0.1
    TD  = 0.1
    iec_curvas = {
        "IEC C1 Standard Inv.":  (0.14, 0.0, 0.02),
        "IEC C2 Very Inv.":      (13.5, 0.0, 1.0),
        "IEC C3 Ext. Inv.":      (80.0, 0.0, 2.0),
    }
    ieee_curvas = {
        "IEEE U1 Moderately Inv.": (0.0515, 0.114, 0.02),
        "IEEE U2 Very Inv.":       (19.61, 0.491, 2.0),
        "IEEE U3 Ext. Inv.":       (28.2, 0.1217, 2.0),
    }
    plt.figure(figsize=(cm_to_inch(16), cm_to_inch(10)))
    for rotulo, (k, c, alpha) in iec_curvas.items():
        plt.plot(M, t_iec(M, TMS, k, c, alpha), label=rotulo, linewidth=1.6)
    for rotulo, (k, c, alpha) in ieee_curvas.items():
        plt.plot(M, t_ieee(M, TD, k, c, alpha), "--", label=rotulo, linewidth=1.6)
    plt.xscale("log")
    plt.yscale("log")
    plt.xlabel(r"Multiplicador de corrente $M = I/I_{pickup}$")
    plt.ylabel("Tempo de operação (s)")
    plt.grid(True, which="both", linestyle="--", alpha=0.6)
    plt.legend(fontsize=8, ncol=2)
    plt.tight_layout()
    plt.savefig(caminho_salvar, format="svg", facecolor="white")
    plt.close()

# ---- Clarke ------------------------------------------------------------------

def clarke_power_invariant(ia, ib, ic):
    k = np.sqrt(2.0 / 3.0)
    i_alpha = k * (ia - 0.5*ib - 0.5*ic)
    i_beta  = k * (np.sqrt(3)/2*ib - np.sqrt(3)/2*ic)
    return i_alpha, i_beta

def plotar_trajetoria_clarke(t, dados_tres_fases, nome_variavel,
                             caminho_salvar, t_ini=None, t_fim=None):
    ia = dados_tres_fases[:, 0]
    ib = dados_tres_fases[:, 1]
    ic = dados_tres_fases[:, 2]
    if (t_ini is not None) and (t_fim is not None):
        mask = (t >= t_ini) & (t <= t_fim)
        ia, ib, ic = ia[mask], ib[mask], ic[mask]
    i_alpha, i_beta = clarke_power_invariant(ia, ib, ic)
    plt.figure(figsize=(cm_to_inch(10), cm_to_inch(10)))
    plt.plot(i_alpha, i_beta, color="black", linewidth=1.3)
    titulo_limpo = nome_variavel.replace("_raw", "").replace("_", " ")
    plt.title(f"Trajetória Clarke (power-invariant)\n{titulo_limpo}")
    plt.xlabel(r"$i_\alpha$ (A)")
    plt.ylabel(r"$i_\beta$ (A)")
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.axis("equal")
    plt.tight_layout()
    plt.savefig(caminho_salvar, format="svg", facecolor="white")
    plt.close()

# ----------------------------------------------------------------------
# 3. FUNÇÕES DE PLOTAGEM DE SINAIS (FFT, RMS, SEQ)
# ----------------------------------------------------------------------

def gerar_grafico_fft_espectro(tempo, dados_raw, nome_arquivo_saida,
                               pasta_saida, ylabel, limite_freq_visual=1000):
    freqs_a, mag_a = calcular_fft_para_plot(dados_raw[:, 0], tempo)
    freqs_b, mag_b = calcular_fft_para_plot(dados_raw[:, 1], tempo)
    freqs_c, mag_c = calcular_fft_para_plot(dados_raw[:, 2], tempo)
    plt.figure()
    plt.plot(freqs_a, mag_a, color="red",   label="Fase A", linewidth=1.2, alpha=0.8)
    plt.plot(freqs_b, mag_b, color="blue",  label="Fase B", linewidth=1.2, alpha=0.8)
    plt.plot(freqs_c, mag_c, color="green", label="Fase C", linewidth=1.2, alpha=0.8)
    plt.xlabel("Frequência (Hz)")
    plt.ylabel(f"Amplitude {ylabel} (Pico)")
    plt.xlim(0, limite_freq_visual)
    plt.legend()
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.tight_layout()
    plt.savefig(pasta_saida / nome_arquivo_saida, format="svg", facecolor="white")
    plt.close()

def plotar_envelope_rms_maximo(tempo, dados_tres_fases, caminho_salvar,
                               label_y="Corrente"):
    imax_vetor = get_imax_envelope_vetor(tempo, dados_tres_fases)
    pico_max_absoluto = np.max(imax_vetor)
    plt.figure(figsize=(cm_to_inch(16), cm_to_inch(10)))
    plt.plot(tempo, imax_vetor, color="red", linewidth=1.5, label="Envelope Máximo")
    plt.xlabel("Tempo (s)")
    plt.ylabel(f"{label_y} Máxima (RMS)")
    plt.title(f"Envelope de {label_y} Máxima")
    props = dict(boxstyle="round", facecolor="white",
                 alpha=0.9, edgecolor="black")
    plt.text(0.5, 0.15, f"Max (RMS): {pico_max_absoluto:.1f}",
             transform=plt.gca().transAxes,
             fontsize=12, color="black", ha="center", bbox=props)
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.tight_layout()
    plt.savefig(caminho_salvar, format="svg", facecolor="white")
    plt.close()

def plotar_rms_tres_fases_com_zoom(tempo, dados_tres_fases, nome_variavel,
                                   caminho_salvar, label_y="Corrente"):
    rms_a = calcular_rms_movel(dados_tres_fases[:, 0], tempo)
    rms_b = calcular_rms_movel(dados_tres_fases[:, 1], tempo)
    rms_c = calcular_rms_movel(dados_tres_fases[:, 2], tempo)
    imax_vetor = np.maximum.reduce([rms_a, rms_b, rms_c])
    idx_pico = np.argmax(imax_vetor)
    t_pico = tempo[idx_pico]
    valor_max_pico = imax_vetor[idx_pico]
    fig = plt.figure(figsize=(cm_to_inch(18), cm_to_inch(16)))
    gs = gridspec.GridSpec(2, 2, height_ratios=[2, 1])
    ax_main = fig.add_subplot(gs[0, :])
    ax_main.plot(tempo, rms_a, color="blue",  label="Fase A", linewidth=1.2)
    ax_main.plot(tempo, rms_b, color="red",   label="Fase B", linewidth=1.2)
    ax_main.plot(tempo, rms_c, color="green", label="Fase C", linewidth=1.2)
    ax_main.axvline(x=t_pico, color="red", linestyle="--", alpha=0.7)
    props = dict(boxstyle="round", facecolor="white",
                 alpha=0.8, edgecolor="gray")
    ax_main.text(0.05, 0.95, f"Máx RMS: {valor_max_pico:.1f}",
                 transform=ax_main.transAxes,
                 fontsize=11, verticalalignment="top",
                 bbox=props, color="black")
    titulo_limpo = nome_variavel.replace("_raw", "").replace("_", " ")
    ax_main.set_title(f"Análise RMS - {titulo_limpo}")
    ax_main.set_ylabel(f"{label_y} RMS")
    ax_main.legend(loc="upper right")
    ax_main.grid(True, linestyle=":", alpha=0.5)
    t_z1_ini, t_z1_fim = max(0, t_pico - 0.15), max(0, t_pico - 0.02)
    t_z2_ini, t_z2_fim = max(0, t_pico - 0.02), min(tempo[-1], t_pico + 0.10)
    ax_z1 = fig.add_subplot(gs[1, 0])
    ax_z1.plot(tempo, rms_a, "b", tempo, rms_b, "r", tempo, rms_c, "g")
    ax_z1.set_xlim(t_z1_ini, t_z1_fim)
    mask1 = (tempo >= t_z1_ini) & (tempo <= t_z1_fim)
    if np.any(mask1):
        y_vals = np.concatenate([rms_a[mask1], rms_b[mask1], rms_c[mask1]])
        ax_z1.set_ylim(np.min(y_vals) * 0.95, np.max(y_vals) * 1.05)
    ax_z1.set_title("Zoom: Pré-Evento", fontsize=10)
    ax_z1.set_xlabel("Tempo (s)")
    ax_z1.set_ylabel(f"{label_y} RMS")
    ax_z1.grid(True, linestyle=":", alpha=0.5)
    ax_z2 = fig.add_subplot(gs[1, 1])
    ax_z2.plot(tempo, rms_a, "b", tempo, rms_b, "r", tempo, rms_c, "g")
    ax_z2.set_xlim(t_z2_ini, t_z2_fim)
    ax_z2.axvline(x=t_pico, color="red", linestyle="--", alpha=0.7)
    mask2 = (tempo >= t_z2_ini) & (tempo <= t_z2_fim)
    if np.any(mask2):
        y_vals_2 = np.concatenate([rms_a[mask2], rms_b[mask2], rms_c[mask2]])
        ax_z2.set_ylim(bottom=0,
                       top=max(valor_max_pico * 1.1, np.max(y_vals_2) * 1.05))
    ax_z2.set_title("Zoom: Evento Principal", fontsize=10)
    ax_z2.set_xlabel("Tempo (s)")
    ax_z2.grid(True, linestyle=":", alpha=0.5)
    plt.tight_layout()
    plt.savefig(caminho_salvar, format="svg", facecolor="white")
    plt.close()

def plotar_sequencias_simetricas(tempo, seq0, seq1, seq2, nome_variavel,
                                 caminho_salvar, label_y="Tensão"):
    plt.figure()
    plt.plot(tempo, seq1, color="blue",  label="Positiva (1)", linewidth=1.5)
    plt.plot(tempo, seq2, color="red",   label="Negativa (2)", linewidth=1.2, linestyle="--")
    plt.plot(tempo, seq0, color="green", label="Zero (0)",     linewidth=1.2, linestyle=":")
    plt.xlabel("Tempo (s)")
    plt.ylabel(f"Magnitude {label_y} (RMS)")
    titulo_limpo = nome_variavel.replace("_raw", "").replace("_", " ")
    plt.title(f"Componentes Simétricas - {titulo_limpo}")
    plt.legend()
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.tight_layout()
    plt.savefig(caminho_salvar, format="svg", facecolor="white")
    plt.close()

# ----------------------------------------------------------------------
# 4. LISTA COMPLETA DE BARRAS E SELEÇÃO
# ----------------------------------------------------------------------

# lista completa possível
TODAS_BARRAS = [
    ("V_800_raw",  "Tensão (V)",   "V_barra_800"),
    ("V_T2F_raw",  "Tensão (V)",   "V_barra_T2F"),
    ("V_T2F1_raw", "Tensão (V)",   "V_barra_T2F1"),
    ("V_818_raw",  "Tensão (V)",   "V_barra_818"),
    ("V_820_raw",  "Tensão (V)",   "V_barra_820"),
    ("V_822_raw",  "Tensão (V)",   "V_barra_822"),
    ("I_800_raw",  "Corrente (A)", "I_barra_800"),
    ("I_T2F_raw",  "Corrente (A)", "I_barra_T2F"),
    ("I_T2F1_raw", "Corrente (A)", "I_barra_T2F1"),
    ("I_818_raw",  "Corrente (A)", "I_barra_818"),
    ("I_820_raw",  "Corrente (A)", "I_barra_820"),
    ("I_822_raw",  "Corrente (A)", "I_barra_822"),
]

# SELECIONE AQUI QUAIS BARRAS QUER PROCESSAR:
# (basta comentar/ descomentar nesta lista)
BARRAS_ATIVAS = [
    "V_800_raw",
    "V_T2F_raw",
    "V_T2F1_raw",
    "V_818_raw",
    "V_820_raw",
    "V_822_raw",
    "I_800_raw",
    "I_T2F_raw",
    "I_T2F1_raw",
    "I_818_raw",
    "I_820_raw",
    "I_822_raw",
]

# cria lista filtrada com base em BARRAS_ATIVAS
todas_vars = [item for item in TODAS_BARRAS if item[0] in BARRAS_ATIVAS]
lista_arquivos_cap5 = [
    r'C:/Users/leosa/OneDrive/Coisas_Leonardo/gits/CurtosT2F/T2F_MATLAB/NovoArtigoPowerDelivery34bus/NovoModeloQualificacao/Teste_Novo_Sem_Terra_14/Processados_HDF5/MRT__Sem_Falta_py.mat',
    r'C:/Users/leosa/OneDrive/Coisas_Leonardo/gits/CurtosT2F/T2F_MATLAB/NovoArtigoPowerDelivery34bus/NovoModeloQualificacao/Teste_Novo_Sem_Terra_14/Processados_HDF5/MRT_SR__Sem_Falta_py.mat',
    r'C:/Users/leosa/OneDrive/Coisas_Leonardo/gits/CurtosT2F/T2F_MATLAB/NovoArtigoPowerDelivery34bus/NovoModeloQualificacao/Teste_Novo_Sem_Terra_14/Processados_HDF5/Qualificacao__Sem_Falta_py.mat',
    r'C:/Users/leosa/OneDrive/Coisas_Leonardo/gits/CurtosT2F/T2F_MATLAB/NovoArtigoPowerDelivery34bus/NovoModeloQualificacao/Teste_Novo_Sem_Terra_14/Processados_HDF5/Qualificacao_SR__Sem_Falta_py.mat',
    r'C:/Users/leosa/OneDrive/Coisas_Leonardo/gits/CurtosT2F/T2F_MATLAB/NovoArtigoPowerDelivery34bus/NovoModeloQualificacao/Teste_Novo_Sem_Terra_14/Processados_HDF5/MRT_sem_terra__Sem_Falta_py.mat',
    r'C:/Users/leosa/OneDrive/Coisas_Leonardo/gits/CurtosT2F/T2F_MATLAB/NovoArtigoPowerDelivery34bus/NovoModeloQualificacao/Teste_Novo_Sem_Terra_14/Processados_HDF5/MRT_SR_sem_terra__Sem_Falta_py.mat',
    r'C:/Users/leosa/OneDrive/Coisas_Leonardo/gits/CurtosT2F/T2F_MATLAB/NovoArtigoPowerDelivery34bus/NovoModeloQualificacao/Teste_Novo_Sem_Terra_14/Processados_HDF5/Qualificacao_sem_terra__Sem_Falta_py.mat',
    r'C:/Users/leosa/OneDrive/Coisas_Leonardo/gits/CurtosT2F/T2F_MATLAB/NovoArtigoPowerDelivery34bus/NovoModeloQualificacao/Teste_Novo_Sem_Terra_14/Processados_HDF5/MRT_sem_terra__A_822_-_Falta_A_py.mat',
    r'C:/Users/leosa/OneDrive/Coisas_Leonardo/gits/CurtosT2F/T2F_MATLAB/NovoArtigoPowerDelivery34bus/NovoModeloQualificacao/Teste_Novo_Sem_Terra_14/Processados_HDF5/MRT_SR__A_822_-_Falta_A_py.mat',
    r'C:/Users/leosa/OneDrive/Coisas_Leonardo/gits/CurtosT2F/T2F_MATLAB/NovoArtigoPowerDelivery34bus/NovoModeloQualificacao/Teste_Novo_Sem_Terra_14/Processados_HDF5/MRT_SR_sem_terra__A_822_-_Falta_A_py.mat',
    r'C:/Users/leosa/OneDrive/Coisas_Leonardo/gits/CurtosT2F/T2F_MATLAB/NovoArtigoPowerDelivery34bus/NovoModeloQualificacao/Teste_Novo_Sem_Terra_14/Processados_HDF5/Qualificacao__R_822_-_Falta_AB_py.mat',
    r'C:/Users/leosa/OneDrive/Coisas_Leonardo/gits/CurtosT2F/T2F_MATLAB/NovoArtigoPowerDelivery34bus/NovoModeloQualificacao/Teste_Novo_Sem_Terra_14/Processados_HDF5/Qualificacao_sem_terra__R_822_-_Falta_AB_py.mat',
    r'C:/Users/leosa/OneDrive/Coisas_Leonardo/gits/CurtosT2F/T2F_MATLAB/NovoArtigoPowerDelivery34bus/NovoModeloQualificacao/Teste_Novo_Sem_Terra_14/Processados_HDF5/Qualificacao_SR__R_822_-_Falta_AB_py.mat',
    r'C:/Users/leosa/OneDrive/Coisas_Leonardo/gits/CurtosT2F/T2F_MATLAB/NovoArtigoPowerDelivery34bus/NovoModeloQualificacao/Teste_Novo_Sem_Terra_14/Processados_HDF5/Qualificacao_SR_sem_terra__R_822_-_Falta_AB_py.mat',
    r'C:/Users/leosa/OneDrive/Coisas_Leonardo/gits/CurtosT2F/T2F_MATLAB/NovoArtigoPowerDelivery34bus/NovoModeloQualificacao/Teste_Novo_Sem_Terra_14/Processados_HDF5/Qualificacao_SR_sem_terra__Sem_Falta_py.mat',
]
# ----------------------------------------------------------------------
# 6. FIGURA GLOBAL TCC E LOOP PRINCIPAL
# ----------------------------------------------------------------------

pasta_tcc = pasta_raiz_resultados / "Curvas_TCC"
pasta_tcc.mkdir(parents=True, exist_ok=True)
plotar_curvas_tcc_religadores(pasta_tcc / "Fig_TCC_IEC_IEEE.svg")

print(f"Iniciando processamento Cap. 5 para {len(lista_arquivos_cap5)} arquivos...\n")

for caminho_str in lista_arquivos_cap5:
    caminho_arquivo = Path(caminho_str)

    try:
        pasta_final = pasta_raiz_resultados / caminho_arquivo.stem
        pasta_final.mkdir(parents=True, exist_ok=True)

        print(f"--> Processando: {caminho_arquivo.name}")

        with h5py.File(caminho_arquivo, "r") as f:
            t = np.array(f["t"]).flatten()

            # 6.1 – sinais por barra selecionada
            for var_mat, label_y_plot, sufixo_nome in todas_vars:
                if var_mat in f:
                    dados_raw = np.array(f[var_mat])

                    if dados_raw.ndim == 2 and dados_raw.shape[0] < dados_raw.shape[1]:
                        dados_raw = dados_raw.T
                    elif dados_raw.ndim == 1:
                        dados_raw = dados_raw.reshape(-1, 1)

                    linhas, colunas = dados_raw.shape
                    dados_padronizados = np.zeros((linhas, 3))
                    for c in range(min(colunas, 3)):
                        dados_padronizados[:, c] = dados_raw[:, c]
                    dados = dados_padronizados

                    if np.max(np.abs(dados)) <= 1e-9:
                        continue

                    # FFT
                    nome_fft = f"fft_{sufixo_nome}.svg"
                    gerar_grafico_fft_espectro(
                        t, dados, nome_fft, pasta_final,
                        label_y_plot, limite_freq_visual=1000
                    )

                    # Barras V1/V3 (só tensões)
                    if var_mat.startswith("V"):
                        nome_barras = f"harmonicas_{sufixo_nome}.svg"
                        v1, v3 = calcular_v1_v3_global(dados[:, 0], t, FREQUENCIA_REDE)
                        from math import isfinite
                        if isfinite(v1) and isfinite(v3):
                            from math import isnan
                            if not isnan(v1):
                                plotar_barras_harmonicas_v1v3(
                                    v1, v3, var_mat,
                                    pasta_final / nome_barras,
                                    label_y_unit="V"
                                )

                    # Envelope RMS
                    nome_env = f"rms_envelope_{sufixo_nome}.svg"
                    plotar_envelope_rms_maximo(
                        t, dados, pasta_final / nome_env,
                        label_y=label_y_plot.split(" ")[0]
                    )

                    # RMS com zoom
                    nome_zoom = f"rms_zoom_{sufixo_nome}.svg"
                    plotar_rms_tres_fases_com_zoom(
                        t, dados, var_mat,
                        pasta_final / nome_zoom,
                        label_y=label_y_plot.split(" ")[0]
                    )

                    # Componentes simétricas
                    seq0, seq1, seq2 = calcular_componentes_simetricas_tempo(
                        dados, t, FREQUENCIA_REDE
                    )
                    nome_seq = f"sequencias_{sufixo_nome}.svg"
                    label_tipo = "Tensão" if "V_" in var_mat else "Corrente"
                    plotar_sequencias_simetricas(
                        t, seq0, seq1, seq2, var_mat,
                        pasta_final / nome_seq,
                        label_y=label_tipo
                    )

                    # Clarke para correntes do T2F
                    if var_mat == "I_T2F_raw" and dados.shape[1] >= 3:
                        nome_clarke = f"clarke_{sufixo_nome}.svg"
                        plotar_trajetoria_clarke(t, dados, var_mat,
                                                 pasta_final / nome_clarke)

        print("    -> OK.")

    except Exception as e:
        print(f"ERRO CRÍTICO ao processar {caminho_arquivo.name}:\n{e}\n")

print("\n--- Processamento Capítulo 5 Finalizado ---")

C:\Users\leosa\AppData\Local\Temp\ipykernel_420056\2103485301.py:110: RuntimeWarning: divide by zero encountered in divide
  return TMS * k / (M**alpha - 1.0) + c
C:\Users\leosa\AppData\Local\Temp\ipykernel_420056\2103485301.py:113: RuntimeWarning: divide by zero encountered in divide
  return TD * k / (M**alpha - 1.0) + c


Iniciando processamento Cap. 5 para 15 arquivos...

--> Processando: MRT__Sem_Falta_py.mat


C:\Users\leosa\AppData\Local\Temp\ipykernel_420056\2103485301.py:270: UserWarning: Attempting to set identical low and high xlims makes transformation singular; automatically expanding.
  ax_z1.set_xlim(t_z1_ini, t_z1_fim)


    -> OK.
--> Processando: MRT_SR__Sem_Falta_py.mat
    -> OK.
--> Processando: Qualificacao__Sem_Falta_py.mat
    -> OK.
--> Processando: Qualificacao_SR__Sem_Falta_py.mat
    -> OK.
--> Processando: MRT_sem_terra__Sem_Falta_py.mat
    -> OK.
--> Processando: MRT_SR_sem_terra__Sem_Falta_py.mat
    -> OK.
--> Processando: Qualificacao_sem_terra__Sem_Falta_py.mat
    -> OK.
--> Processando: MRT_sem_terra__A_822_-_Falta_A_py.mat
    -> OK.
--> Processando: MRT_SR__A_822_-_Falta_A_py.mat
    -> OK.
--> Processando: MRT_SR_sem_terra__A_822_-_Falta_A_py.mat
    -> OK.
--> Processando: Qualificacao__R_822_-_Falta_AB_py.mat
    -> OK.
--> Processando: Qualificacao_sem_terra__R_822_-_Falta_AB_py.mat
    -> OK.
--> Processando: Qualificacao_SR__R_822_-_Falta_AB_py.mat
    -> OK.
--> Processando: Qualificacao_SR_sem_terra__R_822_-_Falta_AB_py.mat
    -> OK.
--> Processando: Qualificacao_SR_sem_terra__Sem_Falta_py.mat
    -> OK.

--- Processamento Capítulo 5 Finalizado ---
